

| Secret name | Value |
|---|---|
| `DB_HOST` | e.g. `ep-xxxx.us-east-2.aws.neon.tech` (from Neon/Supabase) |
| `DB_PORT` | usually `5432` |
| `DB_NAME` | your database name |
| `DB_USER` | your database user |
| `DB_PASSWORD` | your database password |
| `JWT_SECRET` | any long random string (generate one in the next cell) |
| `SMTP_EMAIL` | your Gmail address |
| `SMTP_APP_PASSWORD` | 16-character Gmail **App Password** (not your real password) |
| `NGROK_AUTHTOKEN` | from https://dashboard.ngrok.com/get-started/your-authtoken |

**Note:** the FastAPI backend added below reuses `JWT_SECRET` — no additional secrets are needed for it.


In [1]:
!pip install -q streamlit psycopg2-binary PyJWT bcrypt python-dotenv email-validator pyngrok fastapi uvicorn python-multipart requests \
    langdetect ftfy emoji deep-translator vaderSentiment spacy pandas matplotlib transformers accelerate torch stopwordsiso reportlab \
    bleach pytest datasets scikit-learn seaborn
!python -m spacy download xx_sent_ud_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 25.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.9/74.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.9 MB/s eta 0:00:00
     ━━━━

In [2]:
from google.colab import userdata

required_secrets = [
    "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "JWT_SECRET", "SMTP_EMAIL", "SMTP_APP_PASSWORD", "NGROK_AUTHTOKEN",
]

values = {}
missing = []
for key in required_secrets:
    try:
        values[key] = userdata.get(key)
    except Exception:
        missing.append(key)

if missing:
    raise RuntimeError(
        f"Missing Colab secrets: {missing}. "
        f"Add them via the key icon in the left sidebar, then re-run this cell."
    )

env_content = f'''DB_HOST={values["DB_HOST"]}
DB_PORT={values["DB_PORT"]}
DB_NAME={values["DB_NAME"]}
DB_USER={values["DB_USER"]}
DB_PASSWORD={values["DB_PASSWORD"]}

JWT_SECRET={values["JWT_SECRET"]}
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL={values["SMTP_EMAIL"]}
SMTP_APP_PASSWORD={values["SMTP_APP_PASSWORD"]}

OTP_EXPIRY_MINUTES=10
'''

with open(".env", "w") as f:
    f.write(env_content)

print("Wrote .env with", len(values), "secrets loaded.")

Wrote .env with 9 secrets loaded.


In [3]:
%%writefile db.py
import os, psycopg2
from psycopg2.extras import RealDictCursor
from contextlib import contextmanager
from dotenv import load_dotenv
load_dotenv()

CFG = dict(host=os.getenv("DB_HOST"), port=os.getenv("DB_PORT", "5432"),
           dbname=os.getenv("DB_NAME"), user=os.getenv("DB_USER"),
           password=os.getenv("DB_PASSWORD"), sslmode="require")

@contextmanager
def cursor(commit=False):
    conn = psycopg2.connect(**CFG)
    cur = conn.cursor(cursor_factory=RealDictCursor)
    try:
        yield cur
        if commit: conn.commit()
    finally:
        cur.close(); conn.close()

def init_db():
    with cursor(commit=True) as cur:
        cur.execute("""CREATE TABLE IF NOT EXISTS users (
            id SERIAL PRIMARY KEY, username VARCHAR(50) UNIQUE, email VARCHAR(255) UNIQUE,
            password_hash VARCHAR(255), is_verified BOOLEAN DEFAULT FALSE,
            role VARCHAR(20) NOT NULL DEFAULT 'employee')""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS role VARCHAR(20) NOT NULL DEFAULT 'employee'""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS failed_login_attempts INTEGER NOT NULL DEFAULT 0""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS locked_until TIMESTAMP""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS created_at TIMESTAMP NOT NULL DEFAULT (NOW() AT TIME ZONE 'Asia/Kolkata')""")
        cur.execute("""ALTER TABLE users ALTER COLUMN created_at SET DEFAULT (NOW() AT TIME ZONE 'Asia/Kolkata')""")
        cur.execute("""CREATE TABLE IF NOT EXISTS otp_codes (
            id SERIAL PRIMARY KEY, email VARCHAR(255), code VARCHAR(6),
            purpose VARCHAR(20), expires_at TIMESTAMP, used BOOLEAN DEFAULT FALSE)""")

        cur.execute("""CREATE TABLE IF NOT EXISTS mood_logs (
            id SERIAL PRIMARY KEY,
            user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            mood_date DATE NOT NULL DEFAULT ((NOW() AT TIME ZONE 'Asia/Kolkata')::date),
            sentiment VARCHAR(20),
            emotion VARCHAR(30),
            compound_score REAL,
            confidence REAL,
            journal_text TEXT,
            source VARCHAR(10) NOT NULL DEFAULT 'manual',
            created_at TIMESTAMP NOT NULL DEFAULT (NOW() AT TIME ZONE 'Asia/Kolkata'))""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS source VARCHAR(10) NOT NULL DEFAULT 'manual'""")
        cur.execute("""ALTER TABLE mood_logs ADD COLUMN IF NOT EXISTS confidence REAL""")
        cur.execute("""ALTER TABLE mood_logs ALTER COLUMN mood_date SET DEFAULT ((NOW() AT TIME ZONE 'Asia/Kolkata')::date)""")
        cur.execute("""ALTER TABLE mood_logs ALTER COLUMN created_at SET DEFAULT (NOW() AT TIME ZONE 'Asia/Kolkata')""")
        cur.execute("""ALTER TABLE users ADD COLUMN IF NOT EXISTS profile_photo BYTEA""")
        cur.execute("""CREATE INDEX IF NOT EXISTS idx_mood_logs_user_date
            ON mood_logs(user_id, mood_date)""")

        cur.execute("""CREATE TABLE IF NOT EXISTS questionnaire_responses (
            id SERIAL PRIMARY KEY,
            user_id INTEGER NOT NULL REFERENCES users(id) ON DELETE CASCADE,
            submitted_at TIMESTAMP NOT NULL DEFAULT (NOW() AT TIME ZONE 'Asia/Kolkata'),
            answers JSONB NOT NULL,
            total_score INTEGER NOT NULL,
            max_score INTEGER NOT NULL,
            category VARCHAR(30) NOT NULL,
            wants_to_talk VARCHAR(10))""")
        cur.execute("""ALTER TABLE questionnaire_responses ALTER COLUMN submitted_at SET DEFAULT (NOW() AT TIME ZONE 'Asia/Kolkata')""")
        cur.execute("""CREATE INDEX IF NOT EXISTS idx_questionnaire_user_date
            ON questionnaire_responses(user_id, submitted_at)""")

MOOD_LABELS = ["Happy", "Neutral", "Sad", "Stress", "Angry", "Fear"]

MOOD_EMOJI = {
    "Happy": "\U0001F60A",
    "Neutral": "\U0001F610",
    "Sad": "\U0001F622",
    "Stress": "\U0001F62B",
    "Angry": "\U0001F620",
    "Fear": "\U0001F628",
}

def save_manual_mood(user_id, mood_label):
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, source)
               VALUES (%s, %s, 'manual')""",
            (user_id, mood_label),
        )

def save_profile_photo(user_id, photo_bytes):
    with cursor(commit=True) as cur:
        cur.execute(
            "UPDATE users SET profile_photo=%s WHERE id=%s",
            (psycopg2.Binary(photo_bytes), user_id),
        )

def get_profile_photo(user_id):
    with cursor() as cur:
        cur.execute("SELECT profile_photo FROM users WHERE id=%s", (user_id,))
        row = cur.fetchone()
        if row and row["profile_photo"] is not None:
            return bytes(row["profile_photo"])
        return None

def save_mood_log(user_id, sentiment, emotion, compound_score, journal_text, confidence=None):
    mood_label = emotion if emotion in MOOD_LABELS else "Neutral"
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO mood_logs (user_id, sentiment, emotion, compound_score, confidence, journal_text, source)
               VALUES (%s, %s, %s, %s, %s, %s, 'nlp')""",
            (user_id, mood_label, emotion, compound_score, confidence, journal_text),
        )

def get_mood_logs_for_month(user_id, year, month):
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (mood_date) mood_date, sentiment, emotion, compound_score, confidence, created_at
               FROM mood_logs
               WHERE user_id = %s
                 AND EXTRACT(YEAR FROM mood_date) = %s
                 AND EXTRACT(MONTH FROM mood_date) = %s
               ORDER BY mood_date, created_at DESC""",
            (user_id, year, month),
        )
        return cur.fetchall()

def get_user_mood_history(user_id, limit=200):
    with cursor() as cur:
        cur.execute(
            """SELECT mood_date, sentiment, emotion, compound_score, confidence, journal_text, source, created_at
               FROM mood_logs
               WHERE user_id = %s
               ORDER BY created_at DESC
               LIMIT %s""",
            (user_id, limit),
        )
        return cur.fetchall()

def get_all_employee_mood_logs(limit_days=30):
    with cursor() as cur:
        cur.execute(
            """SELECT u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.compound_score, m.confidence, m.created_at
               FROM mood_logs m
               JOIN users u ON u.id = m.user_id
               WHERE u.role = 'employee'
                 AND m.mood_date >= ((NOW() AT TIME ZONE 'Asia/Kolkata')::date) - (%s || ' days')::interval
               ORDER BY m.mood_date DESC, u.username""",
            (limit_days,),
        )
        return cur.fetchall()

def get_latest_mood_per_employee():
    with cursor() as cur:
        cur.execute(
            """SELECT DISTINCT ON (u.id) u.username, u.email, m.mood_date, m.sentiment, m.emotion, m.confidence, m.created_at
               FROM users u
               JOIN mood_logs m ON m.user_id = u.id
               WHERE u.role = 'employee'
               ORDER BY u.id, m.created_at DESC"""
        )
        return cur.fetchall()

QUESTIONNAIRE_QUESTIONS = [
    {"id": "q1_current_mood", "text": "How are you feeling right now?", "scored": False,
     "options": ["Very Happy", "Happy", "Neutral", "Sad", "Angry", "Anxious/Fearful"]},

    {"id": "q2_overall_mood", "text": "How would you rate your overall mood today?", "scored": True, "reverse": False,
     "options": ["1 - Very Poor", "2 - Poor", "3 - Average", "4 - Good", "5 - Excellent"]},

    {"id": "q3_stress", "text": "How stressed do you feel right now?", "scored": True, "reverse": True,
     "options": ["Not stressed", "Slightly stressed", "Moderately stressed", "Highly stressed", "Extremely stressed"]},

    {"id": "q4_main_factor", "text": "What is the main thing affecting your mood today?", "scored": False,
     "options": ["Work", "Relationships", "Financial concerns", "Health", "Family",
                 "Sleep/Fatigue", "Personal concerns", "Nothing specific", "Other"]},

    {"id": "q5_energy", "text": "How would you describe your energy level today?", "scored": True, "reverse": False,
     "options": ["Very Low", "Low", "Moderate", "High", "Very High"]},

    {"id": "q6_sleep", "text": "How well did you sleep recently?", "scored": True, "reverse": False,
     "options": ["Very Poor", "Poor", "Average", "Good", "Very Good"]},

    {"id": "q7_support_pref", "text": "What kind of support would you prefer right now?", "scored": False,
     "options": ["Breathing/Relaxation Exercise", "Journaling Prompt", "Motivational Content",
                 "Cognitive Reframing", "Mindfulness Activity", "Professional Support Information"]},

    {"id": "q8_want_to_talk", "text": "Would you like to talk about what is bothering you?", "scored": False,
     "options": ["Yes", "Maybe", "No"]},

    {"id": "q9_negative_freq", "text": "How often have you been experiencing negative emotions recently?",
     "scored": True, "reverse": True,
     "options": ["Never", "Rarely", "Sometimes", "Often", "Very Often"]},

    {"id": "q10_confidence", "text": "How confident are you in managing your emotions today?",
     "scored": True, "reverse": False,
     "options": ["Not confident", "Slightly confident", "Moderately confident", "Very confident", "Extremely confident"]},

    {"id": "q11_help_now", "text": "What would help you feel better right now?", "scored": False,
     "options": ["Relaxation", "Someone to talk to", "Motivation", "Taking a break",
                 "Organizing my tasks", "Physical activity", "Sleep/rest", "I'm not sure"]},

    {"id": "q12_personalize", "text": "Would you like MoodMentor to personalize future recommendations based on your answers?",
     "scored": False, "options": ["Yes", "No"]},
]

_SCORED_QUESTIONS = [q for q in QUESTIONNAIRE_QUESTIONS if q["scored"]]
QUESTIONNAIRE_MAX_SCORE = len(_SCORED_QUESTIONS) * 5
QUESTIONNAIRE_MIN_SCORE = len(_SCORED_QUESTIONS) * 1

def score_questionnaire(answers: dict) -> dict:
    total = 0
    for q in _SCORED_QUESTIONS:
        value = q["options"].index(answers[q["id"]]) + 1
        total += (6 - value) if q["reverse"] else value

    if total >= 24:
        category = "Thriving"
    elif total >= 18:
        category = "Doing Well"
    elif total >= 12:
        category = "Needs Attention"
    else:
        category = "At Risk"

    return {"total_score": total, "max_score": QUESTIONNAIRE_MAX_SCORE, "category": category}

def save_questionnaire_response(user_id, answers: dict, total_score: int, category: str):
    import json as _json
    wants_to_talk = answers.get("q8_want_to_talk")
    with cursor(commit=True) as cur:
        cur.execute(
            """INSERT INTO questionnaire_responses (user_id, answers, total_score, max_score, category, wants_to_talk)
               VALUES (%s, %s, %s, %s, %s, %s)""",
            (user_id, _json.dumps(answers), total_score, QUESTIONNAIRE_MAX_SCORE, category, wants_to_talk),
        )

def get_questionnaire_history(user_id, limit=100):
    with cursor() as cur:
        cur.execute(
            """SELECT submitted_at, answers, total_score, max_score, category, wants_to_talk
               FROM questionnaire_responses
               WHERE user_id = %s
               ORDER BY submitted_at DESC
               LIMIT %s""",
            (user_id, limit),
        )
        return cur.fetchall()

def get_all_questionnaire_responses(limit_days=30):
    with cursor() as cur:
        cur.execute(
            """SELECT u.username, u.email, q.submitted_at, q.answers,
                      q.total_score, q.max_score, q.category, q.wants_to_talk
               FROM questionnaire_responses q
               JOIN users u ON u.id = q.user_id
               WHERE u.role = 'employee'
                 AND q.submitted_at >= ((NOW() AT TIME ZONE 'Asia/Kolkata')::date) - (%s || ' days')::interval
               ORDER BY q.submitted_at DESC, u.username""",
            (limit_days,),
        )
        return cur.fetchall()


Writing db.py


In [4]:
from db import cursor

with cursor(commit=True) as cur:
    cur.execute("UPDATE mood_logs SET sentiment = 'Happy' WHERE sentiment = 'Amazing'")
    cur.execute("UPDATE mood_logs SET sentiment = 'Neutral' WHERE sentiment = 'Normal'")
print("Remapped legacy Amazing/Normal rows to Happy/Neutral.")

Remapped legacy Amazing/Normal rows to Happy/Neutral.


In [5]:
%%writefile recommendations.py
WELLNESS_RECOMMENDATIONS = {
    "Happy": [
        "Great to see you're feeling good! Take a moment to note what contributed to this — it helps to recognize your own positive patterns.",
        "Keep this momentum going: consider sharing your positive energy with a colleague or teammate today.",
    ],
    "Neutral": [
        "A calm, steady mood is a good baseline. A short 5-minute walk or stretch break can help maintain it.",
        "Nothing urgent here — this could be a good time to plan your day or check in on a personal goal.",
    ],
    "Sad": {
        "low": "It looks like there might be a touch of sadness here. Consider writing a bit more in your journal about what's on your mind.",
        "medium": "Try a short guided breathing exercise (4 seconds in, 4 seconds hold, 4 seconds out) or step outside for a few minutes.",
        "high": "This seems like a strong low mood. Please consider talking to a trusted colleague, friend, or your HR/EAP wellness contact today.",
    },
    "Stress": {
        "low": "A little stress is normal — try a quick 2-minute breathing break before your next task.",
        "medium": "Consider breaking your current task into smaller steps, and take a 10-minute break away from your screen.",
        "high": "Your stress signal looks high. Try a longer break, deep breathing, or a short walk, and consider flagging your workload to your manager or HR.",
    },
    "Angry": {
        "low": "A bit of frustration is showing. A short pause before responding to anything stressful can help.",
        "medium": "Try stepping away for 5-10 minutes before continuing. Cognitive reframing — writing down the situation objectively — can help too.",
        "high": "This reads as strong frustration or anger. Please take a proper break away from the trigger, and consider talking it through with someone you trust or your HR/EAP contact.",
    },
    "Fear": {
        "low": "A little anxiety is showing. Grounding techniques (naming 5 things you can see, 4 you can hear) can help settle it.",
        "medium": "Try a short guided breathing or grounding exercise, and write down specifically what's worrying you — it often feels more manageable on paper.",
        "high": "This looks like a strong fear/anxiety signal. Please consider reaching out to a trusted colleague, your HR/EAP program, or a mental health professional.",
    },
}

MOOD_TO_EMOTION_BUCKET = {
    "Amazing": "Happy",
    "Happy": "Happy",
    "Normal": "Neutral",
    "Sad": "Sad",
    "Angry": "Angry",
}

def _confidence_bucket(confidence: float) -> str:
    if confidence is None:
        return "medium"
    if confidence < 0.4:
        return "low"
    if confidence < 0.7:
        return "medium"
    return "high"

def get_recommendation(
    emotion_label: str,
    confidence: float = None,
    sentiment: str = None,
    sentiment_score: float = None,
) -> str:
    effective_label = emotion_label
    effective_confidence = confidence

    if emotion_label == "Neutral" and sentiment == "Negative":

        effective_label = "Sad"
        magnitude = abs(sentiment_score) if sentiment_score is not None else 0.3
        effective_confidence = magnitude

    entry = WELLNESS_RECOMMENDATIONS.get(effective_label)
    if entry is None:
        return "Take a moment to check in with yourself today."

    if isinstance(entry, list):

        import random
        return random.choice(entry)

    bucket = _confidence_bucket(effective_confidence)
    return entry[bucket]

def get_period_recommendation(entries: list[dict]) -> str:
    if not entries:
        return "No entries were logged in this period yet."

    bucket_counts: dict[str, int] = {}
    bucket_confidences: dict[str, list[float]] = {}

    for e in entries:
        if e.get("source") == "nlp" and e.get("emotion"):
            bucket = e["emotion"]
            conf = e.get("confidence")
        else:
            bucket = MOOD_TO_EMOTION_BUCKET.get(e.get("sentiment"), "Neutral")
            conf = None

        bucket_counts[bucket] = bucket_counts.get(bucket, 0) + 1
        if conf is not None:
            bucket_confidences.setdefault(bucket, []).append(conf)

    total = sum(bucket_counts.values())
    dominant_bucket = max(bucket_counts, key=bucket_counts.get)
    dominant_count = bucket_counts[dominant_bucket]
    pct = round(100 * dominant_count / total)

    confs = bucket_confidences.get(dominant_bucket)
    avg_conf = sum(confs) / len(confs) if confs else None

    tip = get_recommendation(dominant_bucket, avg_conf)

    overview = (
        f"Over this period, {dominant_bucket.lower()} was your most common state "
        f"({dominant_count} of {total} entries, {pct}%)."
    )
    closing = "Keep logging regularly so trends like this are easier to catch early."

    return f"{overview} {tip} {closing}"

QUESTIONNAIRE_RECOMMENDATIONS = {
    "Thriving": "Your check-in looks great today. Take a moment to note what's working well so you can lean on it again later.",
    "Doing Well": "You're in a solid place overall. A short break or a few minutes of stretching can help you keep this steady.",
    "Needs Attention": "A few areas could use some care today. A proper break or a short walk could help before you push on.",
    "At Risk": "Several signals here point to a tough day. Please consider talking to a trusted colleague, your manager, or your HR/EAP wellness contact -- you don't have to carry this alone.",
}

SUPPORT_PREF_TIPS = {
    "Breathing/Relaxation Exercise": "Try a slow 4-7-8 breathing cycle for two minutes: in for 4, hold for 7, out for 8.",
    "Journaling Prompt": "Try writing for 5 minutes on: \"What's one thing weighing on me, and one thing going right?\"",
    "Motivational Content": "Remember: progress doesn't have to be dramatic today -- one small completed task counts.",
    "Cognitive Reframing": "Ask yourself: is there another way to view today's biggest stressor that feels less all-or-nothing?",
    "Mindfulness Activity": "Try a 3-minute body scan: notice your feet, then breath, then shoulders, releasing tension as you go.",
    "Professional Support Information": "Consider reaching out to your EAP (Employee Assistance Program) or a licensed counselor -- it's confidential and free for most employees.",
}

HELP_NOW_TIPS = {
    "Relaxation": "Step away for 5 minutes and do something calming -- music, stretching, or just sitting quietly.",
    "Someone to talk to": "Reach out to a colleague, friend, or your manager for even a short conversation today.",
    "Motivation": "Break your next task into one small, concrete step and start with just that.",
    "Taking a break": "Block off a proper 10-15 minute break away from your desk this afternoon.",
    "Organizing my tasks": "Spend 5 minutes listing today's tasks in priority order -- it often shrinks the overwhelm.",
    "Physical activity": "A short walk or a few minutes of movement can help reset your energy and mood.",
    "Sleep/rest": "Prioritize winding down earlier tonight -- even 30 extra minutes of sleep helps.",
    "I'm not sure": "That's okay -- sometimes just naming how you feel is a good first step.",
}

def get_questionnaire_recommendation(category: str, support_pref: str | None = None,
                                      help_now: str | None = None, wants_to_talk: str | None = None) -> dict:
    lines = [QUESTIONNAIRE_RECOMMENDATIONS.get(category, "Take a moment to check in with yourself today.")]

    support_tip = SUPPORT_PREF_TIPS.get(support_pref)
    if support_tip:
        lines.append(f"**Since you'd like {support_pref.lower()}:** {support_tip}")

    help_tip = HELP_NOW_TIPS.get(help_now)
    if help_tip:
        lines.append(f"**To help you feel better right now:** {help_tip}")

    escalate = (category == "At Risk") or (wants_to_talk == "Yes")
    if escalate:
        lines.append(
            "It looks like this might be a good moment to talk to someone -- your manager, "
            "HR, or an EAP counselor can help."
        )

    return {"message": "\n\n".join(lines), "escalate": escalate}

def get_team_recommendation(mood_rows: list, questionnaire_rows: list) -> dict:
    from collections import Counter

    stats = {}
    mood_counts = Counter(r["sentiment"] for r in mood_rows if r.get("sentiment"))
    stats["mood_counts"] = dict(mood_counts)

    emo_counts = Counter(r["emotion"] for r in mood_rows if r.get("source") == "nlp" and r.get("emotion"))
    stats["emotion_counts"] = dict(emo_counts)

    total_qn = len(questionnaire_rows)
    category_counts = Counter(r["category"] for r in questionnaire_rows)
    stats["category_counts"] = dict(category_counts)

    factor_counts = Counter()
    support_counts = Counter()
    for r in questionnaire_rows:
        ans = r.get("answers") or {}
        if ans.get("q4_main_factor"):
            factor_counts[ans["q4_main_factor"]] += 1
        if ans.get("q7_support_pref"):
            support_counts[ans["q7_support_pref"]] += 1
    stats["factor_counts"] = dict(factor_counts)
    stats["support_counts"] = dict(support_counts)

    if total_qn == 0:
        return {"message": "Not enough check-in data yet to generate a team recommendation.", "stats": stats}

    at_risk_pct = round(100 * category_counts.get("At Risk", 0) / total_qn)
    doing_well_pct = round(100 * (category_counts.get("Thriving", 0) + category_counts.get("Doing Well", 0)) / total_qn)
    wants_to_talk_count = sum(1 for r in questionnaire_rows if r.get("wants_to_talk") == "Yes")
    top_factor = factor_counts.most_common(1)[0][0] if factor_counts else None

    lines = [
        f"Across {total_qn} check-in(s) this period, {doing_well_pct}% were 'Thriving' or "
        f"'Doing Well', while {at_risk_pct}% were flagged 'At Risk'."
    ]
    if top_factor:
        lines.append(
            f"The most commonly cited factor affecting mood was **{top_factor}** -- worth "
            f"checking whether a team-wide adjustment could help."
        )
    if wants_to_talk_count > 0:
        lines.append(
            f"{wants_to_talk_count} check-in(s) indicated a wish to talk to someone. Individual "
            f"identities aren't shared here -- consider a general reminder about EAP/support "
            f"resources for the whole team rather than singling anyone out."
        )
    if at_risk_pct >= 25:
        lines.append("With a notable share of the team at risk, a lighter workload week or an "
                      "open office-hours slot could help.")

    return {"message": "\n\n".join(lines), "stats": stats}


Writing recommendations.py


In [6]:
%%writefile auth.py
import os, jwt, bcrypt, random, string
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
from db import cursor
load_dotenv()

SECRET = os.getenv("JWT_SECRET")

MAX_LOGIN_ATTEMPTS = 5
LOCKOUT_MINUTES = 15

def hash_pw(pw): return bcrypt.hashpw(pw.encode(), bcrypt.gensalt()).decode()
def check_pw(pw, h): return bcrypt.checkpw(pw.encode(), h.encode())

def make_token(user):
    payload = {"id": user["id"], "username": user["username"], "email": user["email"],
               "role": user.get("role", "employee"),
               "exp": datetime.now(timezone.utc) + timedelta(hours=1)}
    return jwt.encode(payload, SECRET, algorithm="HS256")

def read_token(token):
    try: return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError: return None

def get_user(email):
    with cursor() as cur:
        cur.execute("SELECT * FROM users WHERE email=%s", (email,))
        return cur.fetchone()

def record_failed_login(email):
    with cursor(commit=True) as cur:
        cur.execute(
            "UPDATE users SET failed_login_attempts = failed_login_attempts + 1 WHERE email=%s",
            (email,),
        )
        cur.execute("SELECT failed_login_attempts FROM users WHERE email=%s", (email,))
        row = cur.fetchone()
        if row and row["failed_login_attempts"] >= MAX_LOGIN_ATTEMPTS:
            lock_until = datetime.now(timezone.utc) + timedelta(minutes=LOCKOUT_MINUTES)
            cur.execute("UPDATE users SET locked_until=%s WHERE email=%s", (lock_until, email))

def reset_failed_login(email):
    with cursor(commit=True) as cur:
        cur.execute(
            "UPDATE users SET failed_login_attempts=0, locked_until=NULL WHERE email=%s",
            (email,),
        )

def is_account_locked(user) -> bool:
    locked_until = user.get("locked_until")
    if not locked_until:
        return False
    now = datetime.now(locked_until.tzinfo) if locked_until.tzinfo else datetime.now()
    return now < locked_until

def username_taken(username):
    with cursor() as cur:
        cur.execute("SELECT 1 FROM users WHERE username=%s", (username,))
        return cur.fetchone() is not None

def create_user(username, email, pw, role="employee"):
    with cursor(commit=True) as cur:
        cur.execute("INSERT INTO users (username,email,password_hash,role) VALUES (%s,%s,%s,%s)",
                    (username, email, hash_pw(pw), role))

def verify_user(email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET is_verified=TRUE WHERE email=%s", (email,))

def set_password(email, pw):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET password_hash=%s WHERE email=%s", (hash_pw(pw), email))

def update_email(user_id, new_email):
    with cursor(commit=True) as cur:
        cur.execute("UPDATE users SET email=%s WHERE id=%s", (new_email, user_id))

def new_otp():
    return "".join(random.choices(string.digits, k=6))

def save_otp(email, code, purpose):
    exp = datetime.now(timezone.utc) + timedelta(minutes=10)
    with cursor(commit=True) as cur:
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE email=%s AND purpose=%s", (email, purpose))
        cur.execute("INSERT INTO otp_codes (email,code,purpose,expires_at) VALUES (%s,%s,%s,%s)",
                    (email, code, purpose, exp))

def check_otp(email, code, purpose):
    with cursor(commit=True) as cur:
        cur.execute("""SELECT * FROM otp_codes WHERE email=%s AND purpose=%s AND used=FALSE
                       ORDER BY id DESC LIMIT 1""", (email, purpose))
        row = cur.fetchone()
        if not row or row["code"] != code:
            return False
        now = datetime.now(row["expires_at"].tzinfo) if row["expires_at"].tzinfo else datetime.now()
        if now > row["expires_at"]:
            return False
        cur.execute("UPDATE otp_codes SET used=TRUE WHERE id=%s", (row["id"],))
        return True


Writing auth.py


In [7]:
%%writefile email_utils.py
import os, smtplib
from email.mime.text import MIMEText
from dotenv import load_dotenv
load_dotenv()

HOST, PORT = "smtp.gmail.com", 587
EMAIL = os.getenv("SMTP_EMAIL")
APP_PW = os.getenv("SMTP_APP_PASSWORD")

def send_otp(to_email, code, purpose):
    subject = "Your Verification Code" if purpose == "signup" else "Your Password Reset Code"
    msg = MIMEText(f"Your code is: {code}\nExpires in 10 minutes.")
    msg["From"], msg["To"], msg["Subject"] = EMAIL, to_email, subject
    try:
        with smtplib.SMTP(HOST, PORT, timeout=15) as s:
            s.starttls()
            s.login(EMAIL, APP_PW)
            s.sendmail(EMAIL, to_email, msg.as_string())
        return True, "sent"
    except Exception as e:
        return False, str(e)


Writing email_utils.py


In [8]:
%%writefile security.py
import bleach

MAX_TEXT_LENGTH = 8000

def sanitize_text(text: str, max_length: int = MAX_TEXT_LENGTH) -> str:
    if not text:
        return text
    cleaned = bleach.clean(text, tags=[], attributes={}, strip=True)
    return cleaned[:max_length]


Writing security.py


In [9]:
%%writefile app.py
import os, re, io, base64, calendar
from datetime import date, datetime, timedelta
import requests, streamlit as st
import matplotlib.pyplot as plt
import seaborn as sns
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from db import (init_db, save_mood_log, save_manual_mood, MOOD_LABELS, MOOD_EMOJI,
                 get_mood_logs_for_month, get_user_mood_history,
                 get_all_employee_mood_logs, get_latest_mood_per_employee,
                 QUESTIONNAIRE_QUESTIONS, score_questionnaire, save_questionnaire_response,
                 get_questionnaire_history, get_all_questionnaire_responses,
                 get_profile_photo, save_profile_photo)
from recommendations import (get_period_recommendation, get_questionnaire_recommendation,
                              get_team_recommendation)
import csv
from auth import (make_token, read_token, get_user, username_taken, create_user,
                   verify_user, set_password, check_pw, new_otp, save_otp, check_otp,
                   record_failed_login, reset_failed_login, is_account_locked,
                   MAX_LOGIN_ATTEMPTS, LOCKOUT_MINUTES, update_email)
from email_utils import send_otp
from security import sanitize_text

IST_OFFSET = timedelta(hours=5, minutes=30)

def now_ist():
    return datetime.utcnow() + IST_OFFSET

def today_ist():
    return now_ist().date()

st.set_page_config(page_title="MoodMentor", layout="wide")

sns.set_theme(style="white", rc={
    "figure.facecolor": "none", "axes.facecolor": "none", "savefig.facecolor": "none",
    "font.family": "sans-serif", "text.color": "#241F3B",
    "axes.labelcolor": "#544E72", "xtick.color": "#544E72", "ytick.color": "#544E72",
})

BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")

MOOD_STYLE = {
    "Happy":   {"emoji": MOOD_EMOJI["Happy"],   "color": "#2ecc71"},
    "Neutral": {"emoji": MOOD_EMOJI["Neutral"], "color": "#3498db"},
    "Sad":     {"emoji": MOOD_EMOJI["Sad"],     "color": "#e67e22"},
    "Stress":  {"emoji": MOOD_EMOJI["Stress"],  "color": "#f1c40f"},
    "Angry":   {"emoji": MOOD_EMOJI["Angry"],   "color": "#e74c3c"},
    "Fear":    {"emoji": MOOD_EMOJI["Fear"],    "color": "#9b59b6"},
}
def style_for(label):
    return MOOD_STYLE.get(label, {"emoji": "", "color": "#bdbdbd"})

MOOD_TO_NUM = {"Happy": 2, "Neutral": 0, "Sad": -1, "Stress": -1, "Angry": -2, "Fear": -2}

def inject_css():
    st.markdown("""
    <style>
    @import url('https://fonts.googleapis.com/css2?family=Sora:wght@600;700;800&family=Plus+Jakarta+Sans:wght@400;500;600;700&display=swap');

    :root{
      --joy:#F3A48A; --sadness:#9C90C7; --anger:#EE7268;
      --fear:#77C3E0; --surprise:#EFBE5C; --disgust:#82CFA0;
      --ink:#241F3B; --ink-soft:#544E72;
      --violet:#7C5CFC; --violet-deep:#5B3FE0; --lilac:#C88CF2;
      --grad-brand: linear-gradient(100deg,#7C5CFC 0%, #A874F0 55%, #D98FE0 100%);
      --glass: rgba(255,255,255,0.62); --glass-strong: rgba(255,255,255,0.78); --glass-border: rgba(255,255,255,0.75);
      --shadow-tight: 0 8px 24px -10px rgba(92,63,224,0.25);
      --shadow-soft: 0 20px 60px -20px rgba(92,63,224,0.28);
      --radius: 22px;
    }

    html, body, [class*="css"]{ font-family:'Plus Jakarta Sans', sans-serif; color:var(--ink); }
    .stApp{
      background: linear-gradient(160deg,#EFEAFB 0%, #F1E9F7 30%, #F6E7EE 62%, #FBEADD 100%);
      background-attachment: fixed;
    }
    h1,h2,h3,h4{ font-family:'Sora', sans-serif !important; color:var(--ink) !important; letter-spacing:-.01em; }

        footer{ visibility:hidden; }

        section[data-testid="stSidebar"]{
      background:rgba(255,255,255,0.55); backdrop-filter:blur(18px);
      border-right:1px solid rgba(255,255,255,0.6);
    }

        section[data-testid="stSidebar"] div[data-testid="stButton"]{ margin-bottom:2px; }

        section[data-testid="stSidebar"] .stButton > button[kind="secondary"]{
      background:transparent !important; color:var(--ink-soft) !important;
      border:1.5px solid transparent !important; box-shadow:none !important;
      border-radius:12px !important; font-weight:600 !important; font-size:13.5px !important;
      padding:9px 14px !important; text-align:left !important; justify-content:flex-start !important;
    }
    section[data-testid="stSidebar"] .stButton > button[kind="secondary"]:hover{
      background:rgba(124,92,252,.08) !important; color:var(--violet-deep) !important;
      transform:none !important;
    }
        section[data-testid="stSidebar"] .stButton > button[kind="primary"]{
      border-radius:12px !important; font-weight:800 !important; font-size:13.5px !important;
      padding:9px 14px !important; text-align:left !important; justify-content:flex-start !important;
    }
    section[data-testid="stSidebar"] .stButton > button[kind="primary"]:hover{ transform:none !important; }

        [class*="st-key-mood_opt_"] button{
      height:88px !important; display:flex !important; flex-direction:column !important;
      align-items:center !important; justify-content:center !important;
      font-size:15px !important; font-weight:700 !important; gap:2px !important;
      white-space:pre-line !important; line-height:1.5 !important;
    }

        .st-key-profile_avatar_btn button{
      width:44px !important; height:44px !important; min-width:44px !important; padding:0 !important;
      border-radius:50% !important; border:2px solid rgba(255,255,255,.9) !important;
      background:var(--grad-brand) !important; color:#fff !important; font-weight:800 !important;
      font-family:'Sora',sans-serif !important; box-shadow:var(--shadow-tight) !important;
      transition:.2s !important;
    }
    .st-key-profile_avatar_btn button:hover{ transform:scale(1.06) !important; }

        .stButton > button, .stFormSubmitButton > button, .stDownloadButton > button{
      background:var(--grad-brand) !important; color:#fff !important; border:none !important;
      border-radius:999px !important; font-weight:700 !important; padding:10px 22px !important;
      box-shadow:var(--shadow-tight); transition:.2s ease !important;
    }
    .stButton > button:hover, .stFormSubmitButton > button:hover, .stDownloadButton > button:hover{
      transform:translateY(-2px); box-shadow:0 14px 30px -10px rgba(92,63,224,.5);
    }
    .stButton > button[kind="secondary"]{
      background:#fff !important; color:var(--violet-deep) !important;
      border:1.5px solid rgba(124,92,252,.35) !important; box-shadow:none;
    }

        .stTextInput input, .stTextArea textarea, .stDateInput input, .stNumberInput input,
    div[data-baseweb="select"] > div{
      background:#fff !important; border:1.5px solid rgba(124,92,252,.18) !important;
      border-radius:13px !important; color:var(--ink) !important;
    }
    .stTextInput input:focus, .stTextArea textarea:focus{
      border-color:var(--violet) !important; box-shadow:0 0 0 4px rgba(124,92,252,.14) !important;
    }

        div[role="radiogroup"]{ gap:6px; }
    div[role="radiogroup"] label{
      background:rgba(255,255,255,.55); border:1.5px solid rgba(124,92,252,.15);
      border-radius:999px !important; padding:8px 16px !important; font-weight:600;
    }

        [data-testid="stCameraInput"], [data-testid="stFileUploader"], [data-testid="stAudioInput"]{
      background:var(--glass); border:1.5px dashed rgba(124,92,252,.35);
      border-radius:var(--radius); padding:14px;
    }

        [data-testid="stMetric"]{
      background:var(--glass); border:1px solid var(--glass-border); border-radius:var(--radius);
      padding:16px 18px; box-shadow:var(--shadow-tight);
    }
    [data-testid="stMetricValue"]{ color:var(--violet-deep) !important; font-family:'Sora',sans-serif !important; }

        [data-testid="stAlert"], [data-testid="stAlertContainer"]{
      border-radius:16px !important; border:1px solid rgba(255,255,255,.6) !important;
      backdrop-filter:blur(6px);
    }

        [data-testid="stProgress"] > div > div{ background:var(--grad-brand) !important; border-radius:999px; }
    [data-testid="stProgress"]{ background:rgba(124,92,252,.12); border-radius:999px; }

        [data-testid="stExpander"], [data-testid="stForm"]{
      background:var(--glass-strong); border:1px solid var(--glass-border) !important;
      border-radius:var(--radius) !important; box-shadow:var(--shadow-soft);
    }
    [data-testid="stDataFrame"]{ border-radius:14px; overflow:hidden; box-shadow:var(--shadow-tight); }

        [data-testid="stChatMessage"]{
      background:var(--glass); border-radius:18px; border:1px solid var(--glass-border);
      box-shadow:var(--shadow-tight);
    }
    [data-testid="stChatInput"] textarea{ border-radius:16px !important; }

        div[data-testid="stVerticalBlockBorderWrapper"]{
      border-radius:var(--radius) !important; border:1px solid var(--glass-border) !important;
      background:var(--glass-strong) !important; box-shadow:var(--shadow-soft);
    }
    </style>
    """, unsafe_allow_html=True)

def render_sidebar_brand():
    st.markdown("""
    <div style="display:flex;align-items:center;gap:10px;padding:4px 2px 18px;">
      <svg width="34" height="34" viewBox="0 0 48 48" fill="none">
        <path d="M24 6c-7 0-12 5-12 11 0 3 1 5 3 7-2 1-3 3-3 6 0 5 4 9 9 9h1v3h4v-3h1c5 0 9-4 9-9 0-3-1-5-3-6 2-2 3-4 3-7 0-6-5-11-12-11z" fill="url(#g1)"/>
        <path d="M24 20c1.5-2 4.5-2 5.8-.2 1.2 1.7.6 3.6-1.3 5.4L24 30l-4.5-4.8c-1.9-1.8-2.5-3.7-1.3-5.4 1.3-1.8 4.3-1.8 5.8.2z" fill="#fff"/>
        <defs><linearGradient id="g1" x1="12" y1="6" x2="36" y2="42">
          <stop stop-color="#7C5CFC"/><stop offset="1" stop-color="#D98FE0"/>
        </linearGradient></defs>
      </svg>
      <div style="line-height:1.1;">
        <div style="font-family:'Sora',sans-serif;font-weight:800;font-size:17px;color:#241F3B;">
          Mood<span style="background:linear-gradient(100deg,#7C5CFC,#D98FE0);-webkit-background-clip:text;background-clip:text;color:transparent;">Mentor</span>
        </div>
        <div style="font-size:9.5px;font-weight:700;letter-spacing:.04em;color:#544E72;">EMOTIONAL WELLNESS</div>
      </div>
    </div>
    """, unsafe_allow_html=True)

def donut_chart(counts: dict, size=2.6):
    labels, values, colors = [], [], []
    for k, v in counts.items():
        if v > 0:
            labels.append(k); values.append(v)
            colors.append(style_for(k)["color"])
    if not values:
        return None
    fig, ax = plt.subplots(figsize=(size, size))
    ax.pie(values, colors=colors, startangle=90, wedgeprops=dict(width=0.38, edgecolor="white"))
    ax.set(aspect="equal")
    fig.patch.set_alpha(0.0)
    return fig

BRAND_PALETTE = ["#7C5CFC", "#A874F0", "#D98FE0", "#77C3E0", "#82CFA0", "#EFBE5C", "#EE7268", "#F3A48A", "#9C90C7"]

def _palette_for(labels):
    return [style_for(l)["color"] if l in MOOD_STYLE else BRAND_PALETTE[i % len(BRAND_PALETTE)]
            for i, l in enumerate(labels)]

def styled_bar_chart(data: dict, figsize=(5.4, 3.1)):
    data = {k: v for k, v in data.items() if v is not None}
    if not data:
        return None
    order = sorted(data.items(), key=lambda kv: kv[1])
    labels = [k for k, _ in order]
    values = [v for _, v in order]
    colors = _palette_for(labels)
    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(x=values, y=labels, hue=labels, palette=colors, dodge=False, legend=False, ax=ax)
    for i, v in enumerate(values):
        label = f"{v:.0f}" if float(v).is_integer() else f"{v:.1f}"
        ax.text(v, i, f"  {label}", va="center", fontsize=9, color="#544E72")
    ax.set_xlim(0, max(values) * 1.2 if max(values) > 0 else 1)
    ax.set_xlabel(""); ax.set_ylabel("")
    sns.despine(bottom=True, left=True, ax=ax)
    ax.tick_params(axis="both", length=0)
    ax.grid(axis="x", color="#7C5CFC", alpha=0.1)
    fig.patch.set_alpha(0.0); ax.patch.set_alpha(0.0)
    fig.tight_layout()
    return fig

def styled_line_chart(data: dict, figsize=(5.8, 3.1)):
    data = {k: v for k, v in data.items() if v is not None}
    if not data:
        return None
    labels = list(data.keys())
    values = list(data.values())
    xs = list(range(len(labels)))
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(xs, values, color="#7C5CFC", linewidth=2.4, marker="o", markersize=4.5,
            markerfacecolor="#7C5CFC", markeredgecolor="white", markeredgewidth=1)
    ax.fill_between(xs, values, min(values), color="#7C5CFC", alpha=0.12)
    step = max(1, len(labels) // 8)
    ax.set_xticks(xs[::step])
    ax.set_xticklabels([labels[i] for i in xs[::step]], rotation=30, ha="right", fontsize=8)
    ax.set_xlabel(""); ax.set_ylabel("")
    sns.despine(ax=ax)
    ax.grid(axis="y", color="#7C5CFC", alpha=0.1)
    fig.patch.set_alpha(0.0); ax.patch.set_alpha(0.0)
    fig.tight_layout()
    return fig

def metric_tile(label, value, sub=None):
    st.metric(label, value, delta=sub, delta_color="off")

def build_pdf_report(username, start_d, end_d, entries, recommendation_text):
    buf = io.BytesIO()
    doc = SimpleDocTemplate(buf, pagesize=letter, topMargin=48, bottomMargin=48)
    styles = getSampleStyleSheet()
    story = []

    story.append(Paragraph("MoodMentor Wellness Report", styles["Title"]))
    story.append(Paragraph(f"{username} &nbsp;|&nbsp; {start_d} to {end_d}", styles["Normal"]))
    story.append(Spacer(1, 16))

    counts = {}
    for h in entries:
        counts[h["sentiment"]] = counts.get(h["sentiment"], 0) + 1
    summary_line = ", ".join(f"{k}: {v}" for k, v in counts.items())
    story.append(Paragraph("Mood summary", styles["Heading2"]))
    story.append(Paragraph(f"{len(entries)} entries logged. {summary_line}.", styles["Normal"]))
    story.append(Spacer(1, 12))

    story.append(Paragraph("Recommendation", styles["Heading2"]))
    story.append(Paragraph(recommendation_text, styles["Normal"]))
    story.append(Spacer(1, 16))

    story.append(Paragraph("Entries", styles["Heading2"]))
    table_data = [["Date", "Time", "Mood", "Emotion", "Confidence", "Source"]]
    for h in sorted(entries, key=lambda r: r["created_at"], reverse=True):
        table_data.append([
            str(h["mood_date"]),
            h["created_at"].strftime("%H:%M"),
            h["sentiment"] or "\u2014",
            h.get("emotion") or "\u2014",
            f"{h['confidence']:.0%}" if h.get("confidence") is not None else "\u2014",
            h["source"],
        ])
    tbl = Table(table_data, repeatRows=1, hAlign="LEFT")
    tbl.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#444444")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTSIZE", (0, 0), (-1, -1), 8),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#dddddd")),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f5f7f6")]),
    ]))
    story.append(tbl)

    doc.build(story)
    buf.seek(0)
    return buf.getvalue()

def build_csv_export(rows: list, fieldnames: list) -> bytes:
    buf = io.StringIO()
    writer = csv.DictWriter(buf, fieldnames=fieldnames, extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        writer.writerow(row)
    return buf.getvalue().encode("utf-8")

def render_profile_section(user, role):
    uid = user["id"]
    photo_key = f"profile_photo_{uid}"
    if photo_key not in st.session_state:
        st.session_state[photo_key] = get_profile_photo(uid)
    edit_key = f"editing_photo_{uid}"
    if edit_key not in st.session_state:
        st.session_state[edit_key] = False

    st.subheader(" Profile")

    col1, col2 = st.columns([1, 3])
    with col1:
        if st.session_state.get(photo_key):
            st.image(st.session_state[photo_key], width=88)
        else:
            initial = (user.get("username") or "?").strip()[0].upper()
            st.write(f"### {initial}")
    with col2:
        st.write(f"**{user.get('username', '—')}**")
        st.caption(role.capitalize())
        if st.button("Change Profile Photo", key="profile_edit_btn"):
            st.session_state[edit_key] = not st.session_state[edit_key]

    if st.session_state[edit_key]:
        st.write("")
        st.markdown("**Upload & Adjust Profile Picture**")
        uploaded_photo = st.file_uploader(
            "Choose an image", type=["png", "jpg", "jpeg"], key=f"photo_uploader_{uid}",
        )
        if uploaded_photo is not None:
            from PIL import Image
            img = Image.open(uploaded_photo).convert("RGB")
            w, h = img.size
            min_side = min(w, h)
            zoom = st.slider("Zoom", 1.0, 3.0, 1.0, 0.05, key=f"zoom_{uid}")
            crop_size = max(10, min(int(min_side / zoom), min_side))
            max_x = max(w - crop_size, 0)
            max_y = max(h - crop_size, 0)
            x = st.slider("Move Horizontal", 0, max_x, max_x // 2, key=f"offx_{uid}") if max_x > 0 else 0
            y = st.slider("Move Vertical", 0, max_y, max_y // 2, key=f"offy_{uid}") if max_y > 0 else 0
            cropped = img.crop((x, y, x + crop_size, y + crop_size)).resize((300, 300))

            pc1, pc2 = st.columns([1, 3])
            with pc1:
                st.image(cropped, width=120, caption="Preview")
            with pc2:
                st.write("")
                save_col, cancel_col = st.columns(2)
                with save_col:
                    if st.button("Save Photo", key=f"save_photo_{uid}", type="primary", use_container_width=True):
                        buf = io.BytesIO()
                        cropped.save(buf, format="PNG")
                        photo_bytes = buf.getvalue()
                        save_profile_photo(uid, photo_bytes)
                        st.session_state[photo_key] = photo_bytes
                        st.session_state[edit_key] = False
                        st.success("Profile picture updated!")
                        st.rerun()
                with cancel_col:
                    if st.button("Cancel", key=f"cancel_photo_{uid}", use_container_width=True):
                        st.session_state[edit_key] = False
                        st.rerun()

    st.write("")
    left, right = st.columns(2)

    with left:
        st.markdown("**User Information**")
        st.write("")
        st.text_input("User ID", value=str(user.get("id", "—")), disabled=True)
        st.text_input("Name", value=user.get("username", "—"), disabled=True)
        st.text_input("Role", value=str(role).capitalize(), disabled=True)

        email_edit_key = f"editing_email_{uid}"
        otp_sent_key = f"email_otp_sent_{uid}"
        pending_email_key = f"pending_new_email_{uid}"
        if email_edit_key not in st.session_state:
            st.session_state[email_edit_key] = False
        if otp_sent_key not in st.session_state:
            st.session_state[otp_sent_key] = False

        ec1, ec2 = st.columns([4, 1.5])
        with ec1:
            st.text_input("Email ID", value=user.get("email", "—"), disabled=True, key=f"email_display_{uid}")
        with ec2:
            st.write("")
            if st.button("Edit Email", key=f"edit_email_btn_{uid}", use_container_width=True):
                st.session_state[email_edit_key] = not st.session_state[email_edit_key]
                st.session_state[otp_sent_key] = False

        if st.session_state[email_edit_key]:
            if not st.session_state[otp_sent_key]:
                new_email = st.text_input("New Email", placeholder="Enter new email", key=f"new_email_input_{uid}")
                sc1, sc2 = st.columns(2)
                with sc1:
                    if st.button("Send OTP", key=f"send_email_otp_{uid}", type="primary", use_container_width=True):
                        new_email_clean = sanitize_text(new_email.strip().lower())
                        if not new_email_clean or "@" not in new_email_clean:
                            st.error("Enter a valid email address.")
                        elif new_email_clean == (user.get("email") or "").lower():
                            st.error("This is already your current email.")
                        elif get_user(new_email_clean):
                            st.error("This email is already in use.")
                        else:
                            code = new_otp()
                            save_otp(new_email_clean, code, "email_change")
                            ok, msg = send_otp(new_email_clean, code, "email_change")
                            if ok:
                                st.session_state[pending_email_key] = new_email_clean
                                st.session_state[otp_sent_key] = True
                                st.success("OTP sent to your new email.")
                                st.rerun()
                            else:
                                st.error(f"Failed to send OTP: {msg}")
                with sc2:
                    if st.button("Cancel", key=f"cancel_email_edit1_{uid}", use_container_width=True):
                        st.session_state[email_edit_key] = False
                        st.rerun()
            else:
                pending_email = st.session_state.get(pending_email_key, "")
                st.caption(f"Enter the code sent to **{pending_email}**")
                otp_code = st.text_input("Verification Code", max_chars=6, key=f"email_otp_code_{uid}")
                vc1, vc2 = st.columns(2)
                with vc1:
                    if st.button("Verify & Update", key=f"verify_email_otp_{uid}", type="primary", use_container_width=True):
                        if check_otp(pending_email, otp_code.strip(), "email_change"):
                            update_email(uid, pending_email)
                            updated_user = get_user(pending_email)
                            st.session_state.token = make_token(updated_user)
                            st.session_state[email_edit_key] = False
                            st.session_state[otp_sent_key] = False
                            st.success("Email updated successfully!")
                            st.rerun()
                        else:
                            st.error("Invalid or expired code.")
                with vc2:
                    if st.button("Cancel", key=f"cancel_email_edit2_{uid}", use_container_width=True):
                        st.session_state[email_edit_key] = False
                        st.session_state[otp_sent_key] = False
                        st.rerun()

    with right:
        st.markdown("**Change Password**")
        st.write("")
        current_pw = st.text_input("Current Password", type="password", placeholder="Enter current password", key=f"pw_current_{uid}")
        new_pw = st.text_input("New Password", type="password", placeholder="Enter new password", key=f"pw_new_{uid}")
        confirm_pw = st.text_input("Confirm Password", type="password", placeholder="Confirm new password", key=f"pw_confirm_{uid}")

        if st.button("Update Password", type="primary", use_container_width=True, key=f"pw_update_btn_{uid}"):
            full = get_user(user["email"])
            stored_hash = full["password_hash"] if full else None
            if not stored_hash or not check_pw(current_pw, stored_hash):
                st.error("Current password is incorrect.")
            elif not valid_pw(new_pw):
                st.error("Password needs 8+ chars, letters and numbers.")
            elif new_pw != confirm_pw:
                st.error("New password and confirm password do not match.")
            else:
                set_password(user["email"], new_pw)
                st.success("Password updated successfully.")

inject_css()

@st.cache_resource
def setup(): init_db()
setup()

if "page" not in st.session_state: st.session_state.page = "welcome"
if "show_auth_panel" not in st.session_state: st.session_state.show_auth_panel = False
if "auth_mode" not in st.session_state: st.session_state.auth_mode = "login"
if "token" not in st.session_state: st.session_state.token = None
if "email" not in st.session_state: st.session_state.email = None
if "chat_history" not in st.session_state: st.session_state.chat_history = []
if "cal_year" not in st.session_state: st.session_state.cal_year = today_ist().year
if "cal_month" not in st.session_state: st.session_state.cal_month = today_ist().month
if "today_mood_saved" not in st.session_state: st.session_state.today_mood_saved = False
if "nav" not in st.session_state: st.session_state.nav = "Home"

def goto_auth(mode): st.session_state.auth_mode = mode; st.rerun()

def valid_pw(pw):
    return len(pw) >= 8 and re.search(r"[A-Za-z]", pw) and re.search(r"[0-9]", pw)

if st.session_state.token:
    user = read_token(st.session_state.token)
    if user:
        role = user.get("role", "employee")
        headers = {"Authorization": f"Bearer {st.session_state.token}"}

        with st.sidebar:
            render_sidebar_brand()
            if role == "employee":
                nav_options = ["Home", "Journal", "Questionnaire", "Wellness Chat",
                                "Face Detection", "Voice Analyzer", "Focus Timer", "Relax",
                                "Dashboard"]
            else:
                nav_options = ["Reports"]
            for _opt in nav_options:
                _is_active = st.session_state.nav == _opt
                if st.button(_opt, key=f"navtab_{_opt}", use_container_width=True,
                             type="primary" if _is_active else "secondary"):
                    st.session_state.nav = _opt
                    st.rerun()
            st.divider()
            st.caption(f"Signed in as **{user['username']}**")
            st.caption(f"{user['email']} · {role.capitalize()}")
            if st.button("Log out", use_container_width=True):
                st.session_state.token = None
                st.session_state.page = "welcome"
                st.session_state.show_auth_panel = False
                st.rerun()

        _now_ist = now_ist()
        greeting = "Good Morning" if _now_ist.hour < 12 else (
            "Good Afternoon" if _now_ist.hour < 18 else "Good Evening")

        header_left, header_right = st.columns([6, 1])
        with header_left:
            st.subheader(f"{greeting}, {user['username']}!")
            st.caption("Here's your emotional wellness overview.")
        with header_right:
            _uid = user["id"]
            _photo_key = f"profile_photo_{_uid}"
            if _photo_key not in st.session_state:
                st.session_state[_photo_key] = get_profile_photo(_uid)
            _photo_bytes = st.session_state.get(_photo_key)
            with st.container(key="profile_avatar_btn"):
                _initial = (user.get("username") or "?").strip()[0].upper()
                if st.button(_initial, key="profile_avatar_inner"):
                    st.session_state.nav = "Profile"
                    st.rerun()
            if _photo_bytes:
                _b64 = base64.b64encode(_photo_bytes).decode()
                st.markdown(
                    f'<style>.st-key-profile_avatar_btn button {{'
                    f'background-image:url("data:image/png;base64,{_b64}") !important;'
                    f'background-size:cover !important; background-position:center !important;'
                    f'color:transparent !important;}}</style>',
                    unsafe_allow_html=True,
                )

        if role == "employee":
            section = st.session_state.nav

            if section == "Home":
                history_all = get_user_mood_history(user["id"], limit=500)
                latest = history_all[0] if history_all else None
                today_count = sum(1 for h in history_all if h["mood_date"] == today_ist())
                streak = 0
                day_ptr = today_ist()
                day_set = {h["mood_date"] for h in history_all}
                while day_ptr in day_set:
                    streak += 1
                    day_ptr = date.fromordinal(day_ptr.toordinal() - 1)

                positive_count = sum(1 for h in history_all if h["sentiment"] == "Happy")
                overall_score = int(100 * positive_count / len(history_all)) if history_all else 0

                m1, m2, m3, m4 = st.columns(4)
                with m1:
                    if latest:
                        s = style_for(latest["sentiment"])
                        metric_tile("Current Mood", f"{s['emoji']} {latest['sentiment']}")
                    else:
                        metric_tile("Current Mood", "—")
                with m2:
                    metric_tile("Overall Score", f"{overall_score}%", "Positive" if overall_score >= 50 else "Needs care")
                with m3:
                    metric_tile("Entries Today", today_count)
                with m4:
                    metric_tile("Current Streak", f"{streak} Days")

                st.write("")
                st.subheader("How Do You Feel?")
                now = now_ist()
                st.caption(f"{now.strftime('%Y-%m-%d')}  {now.strftime('%H:%M')}")

                cols = st.columns(len(MOOD_LABELS))
                picked = st.session_state.get("picked_mood")
                for col, label in zip(cols, MOOD_LABELS):
                    s = style_for(label)
                    is_sel = picked == label
                    with col:
                        with st.container(key=f"mood_opt_{label}"):
                            if st.button(f"{s['emoji']}\n{label}", key=f"pick_{label}",
                                         use_container_width=True,
                                         type="primary" if is_sel else "secondary"):
                                st.session_state.picked_mood = label

                st.write("")
                confirm_col = st.columns([3, 1, 3])[1]
                with confirm_col:
                    disabled = picked is None
                    if st.button("Save mood", type="primary", disabled=disabled,
                                 use_container_width=True):
                        save_manual_mood(user["id"], st.session_state.picked_mood)
                        st.session_state.today_mood_saved = True
                        st.session_state.picked_mood = None
                        st.rerun()

                if st.session_state.today_mood_saved:
                    st.success("Today's mood saved!")
                    st.session_state.today_mood_saved = False

                st.subheader("Your Mood Calendar")

                nav_l, nav_mid, nav_r = st.columns([1, 3, 1])
                if nav_l.button("← Prev"):
                    m, y = st.session_state.cal_month - 1, st.session_state.cal_year
                    if m == 0: m, y = 12, y - 1
                    st.session_state.cal_month, st.session_state.cal_year = m, y
                    st.rerun()
                if nav_r.button("Next →"):
                    m, y = st.session_state.cal_month + 1, st.session_state.cal_year
                    if m == 13: m, y = 1, y + 1
                    st.session_state.cal_month, st.session_state.cal_year = m, y
                    st.rerun()
                nav_mid.markdown(
                    f"**{calendar.month_name[st.session_state.cal_month]} "
                    f"{st.session_state.cal_year}**"
                )

                logs = get_mood_logs_for_month(user["id"], st.session_state.cal_year,
                                                st.session_state.cal_month)
                by_day = {row["mood_date"].day: row for row in logs}

                weeks = calendar.Calendar(firstweekday=6).monthdayscalendar(
                    st.session_state.cal_year, st.session_state.cal_month
                )
                day_names = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
                header_cols = st.columns(7)
                for c, name in zip(header_cols, day_names):
                    c.markdown(f"**{name}**")

                for week in weeks:
                    cols = st.columns(7)
                    for col, day_num in zip(cols, week):
                        if day_num == 0:
                            col.write("")
                            continue
                        entry = by_day.get(day_num)
                        s = style_for(entry["sentiment"] if entry else None)
                        time_label = entry["created_at"].strftime("%H:%M") if entry else ""
                        col.write(f"{day_num}")
                        col.write(s["emoji"] if entry else "")
                        col.caption(time_label)

                legend = " · ".join(l for l in MOOD_LABELS)
                st.caption(f"{legend} · No entry logged  (hover/see time under each day)")

            elif section == "Journal":
                st.subheader(" Journal")
                journal_text = st.text_area(
                    "Write about how you're feeling today", height=150,
                    placeholder="Your note here...",
                )
                if st.button("Analyze my entry"):
                    if not journal_text.strip():
                        st.warning("Write something first.")
                    else:
                        clean_text = sanitize_text(journal_text.strip())
                        with st.spinner("Running NLP analysis…"):
                            try:
                                resp = requests.post(
                                    f"{BACKEND_URL}/analyze-text",
                                    json={"text": clean_text},
                                    headers=headers, timeout=120,
                                )
                            except requests.exceptions.RequestException as e:
                                st.error(f"Could not reach backend: {e}"); resp = None
                        if resp is not None:
                            if resp.status_code != 200:
                                st.error("Analysis failed.")
                            else:
                                r = resp.json()
                                confidence = r.get("emotion_confidence")
                                save_mood_log(
                                    user["id"], r["final_sentiment"], r["final_emotion"],
                                    r["sentiment_scores"]["compound"], clean_text,
                                    confidence=confidence,
                                )
                                conf_str = f", Confidence: **{confidence:.0%}**" if confidence is not None else ""
                                st.success(f"Saved! Sentiment: **{r['final_sentiment']}**, "
                                           f"Emotion: **{r['final_emotion']}**{conf_str}")
                                _fig = styled_bar_chart(r["emotion_scores"])
                                if _fig: st.pyplot(_fig, use_container_width=True)
                                if r.get("recommendation"):
                                    st.info(f"**Recommendation:** {r['recommendation']}")

                st.subheader("Or upload a file")
                uploaded = st.file_uploader("Choose a CSV or TXT file", type=["csv", "txt"])
                if uploaded is not None and st.button("Run NLP Analysis on file"):
                    files = {"file": (uploaded.name, uploaded.getvalue())}
                    with st.spinner("Running multilingual NLP pipeline…"):
                        try:
                            resp = requests.post(f"{BACKEND_URL}/analyze", files=files,
                                                  headers=headers, timeout=120)
                        except requests.exceptions.RequestException as e:
                            st.error(f"Could not reach backend: {e}"); resp = None
                    if resp is not None:
                        if resp.status_code != 200:
                            st.error("Analysis failed.")
                        else:
                            r = resp.json()
                            confidence = r.get("emotion_confidence")
                            save_mood_log(
                                user["id"], r["final_sentiment"], r["final_emotion"],
                                r["sentiment_scores"]["compound"], r.get("cleaned_text", ""),
                                confidence=confidence,
                            )
                            conf_str = f", Confidence: **{confidence:.0%}**" if confidence is not None else ""
                            st.success(f"Saved! Sentiment: **{r['final_sentiment']}**, "
                                       f"Emotion: **{r['final_emotion']}**{conf_str}")
                            _fig = styled_bar_chart(r["emotion_scores"])
                            if _fig: st.pyplot(_fig, use_container_width=True)
                            if r.get("recommendation"):
                                st.info(f"**Recommendation:** {r['recommendation']}")

                st.subheader(" Past entries")
                history = [h for h in get_user_mood_history(user["id"], limit=20)
                           if h["journal_text"]]
                if not history:
                    st.caption("No journal entries yet.")
                for h in history:
                    s = style_for(h["sentiment"])
                    conf_str = f" · Confidence: {h['confidence']:.0%}" if h.get("confidence") is not None else ""
                    with st.expander(
                        f"{s['emoji']} {h['sentiment']} — {h['created_at'].strftime('%Y-%m-%d %H:%M')}{conf_str}"
                    ):
                        st.write(h["journal_text"])

            elif section == "Questionnaire":
                st.subheader(" Wellness Check-in")
                st.caption(
                    "A quick 12-question check-in. Your ratings feed a short wellness score; "
                    "the rest just helps tailor the suggestion you get afterward."
                )

                with st.form("questionnaire_form"):
                    answers = {}
                    for q in QUESTIONNAIRE_QUESTIONS:
                        answers[q["id"]] = st.radio(
                            q["text"], q["options"], key=f"qn_{q['id']}", horizontal=False,
                        )
                    submitted = st.form_submit_button("Submit check-in")

                if submitted:
                    result = score_questionnaire(answers)
                    save_questionnaire_response(
                        user["id"], answers, result["total_score"], result["category"],
                    )
                    rec = get_questionnaire_recommendation(
                        result["category"],
                        support_pref=answers.get("q7_support_pref"),
                        help_now=answers.get("q11_help_now"),
                        wants_to_talk=answers.get("q8_want_to_talk"),
                    )
                    st.success(
                        f"Check-in saved! Wellness score: **{result['total_score']}/{result['max_score']}** "
                        f"— **{result['category']}**"
                    )
                    if rec["escalate"]:
                        st.warning(rec["message"])
                    else:
                        st.info(rec["message"])

                st.subheader(" Past check-ins")
                qn_history = get_questionnaire_history(user["id"], limit=50)
                if not qn_history:
                    st.caption("No check-ins yet.")
                else:
                    qn_table = [{
                        "Date": h["submitted_at"].strftime("%Y-%m-%d %H:%M"),
                        "Score": f"{h['total_score']}/{h['max_score']}",
                        "Category": h["category"],
                        "Wants to talk?": h.get("wants_to_talk") or "—",
                    } for h in qn_history]
                    st.dataframe(qn_table, use_container_width=True)

                    if st.button("Export CSV", key="qn_export_csv"):
                        csv_rows = []
                        for h in qn_history:
                            row = {
                                "submitted_at": h["submitted_at"].strftime("%Y-%m-%d %H:%M"),
                                "total_score": h["total_score"], "max_score": h["max_score"],
                                "category": h["category"],
                            }
                            row.update(h["answers"])
                            csv_rows.append(row)
                        fieldnames = ["submitted_at", "total_score", "max_score", "category"] +                                     [q["id"] for q in QUESTIONNAIRE_QUESTIONS]
                        csv_bytes = build_csv_export(csv_rows, fieldnames)
                        st.download_button(
                            "Download CSV", data=csv_bytes,
                            file_name=f"moodmentor_questionnaire_{user['username']}.csv",
                            mime="text/csv",
                        )

            elif section == "Wellness Chat":
                st.subheader(" Wellness Chat")
                st.caption("A supportive space to talk about how you're feeling. "
                           "Not a substitute for professional care.")
                chat_box = st.container(height=450)
                with chat_box:
                    for turn in st.session_state.chat_history:
                        with st.chat_message(turn["role"]):
                            st.write(turn["content"])

                user_msg = st.chat_input("How are you feeling today?")
                if user_msg:
                    user_msg = sanitize_text(user_msg)
                    st.session_state.chat_history.append({"role": "user", "content": user_msg})
                    recent_history = st.session_state.chat_history[-10:-1]
                    try:
                        resp = requests.post(
                            f"{BACKEND_URL}/chat",
                            json={"message": user_msg, "history": recent_history},
                            headers=headers, timeout=60,
                        )
                        reply = resp.json()["reply"] if resp.status_code == 200 else                            "Sorry, I couldn't reach the wellness assistant right now."
                    except requests.exceptions.RequestException:
                        reply = "Sorry, I couldn't reach the wellness assistant right now."
                    st.session_state.chat_history.append({"role": "assistant", "content": reply})
                    st.rerun()

                if st.session_state.chat_history and st.button("Clear chat"):
                    st.session_state.chat_history = []
                    st.rerun()

            elif section == "Dashboard":
                history = get_user_mood_history(user["id"], limit=200)
                if not history:
                    st.info("No entries yet — pick a mood on Home or write a journal entry to see your dashboard.")
                else:
                    counts = {label: 0 for label in MOOD_LABELS}
                    for h in history:
                        if h["sentiment"] in counts:
                            counts[h["sentiment"]] += 1

                    c1, c2 = st.columns(2)
                    with c1:
                        st.write("**Mood distribution**")
                        fig = donut_chart(counts)
                        if fig: st.pyplot(fig, use_container_width=False)
                        else:
                            _fig = styled_bar_chart(counts)
                            if _fig: st.pyplot(_fig, use_container_width=True)
                    with c2:
                        st.write("**Mood trend over time**")
                        by_date = {}
                        for h in history:
                            d = h["mood_date"]
                            by_date.setdefault(d, []).append(MOOD_TO_NUM.get(h["sentiment"], 0))
                        trend = {str(d): sum(v) / len(v) for d, v in sorted(by_date.items())}
                        _fig = styled_line_chart(trend)
                        if _fig: st.pyplot(_fig, use_container_width=True)

                    st.write("**Emotions detected from journal entries**")
                    emo_counts = {}
                    for h in history:
                        if h["source"] == "nlp" and h["emotion"]:
                            emo_counts[h["emotion"]] = emo_counts.get(h["emotion"], 0) + 1
                    if emo_counts:
                        _fig = styled_bar_chart(emo_counts)
                        if _fig: st.pyplot(_fig, use_container_width=True)
                    else:
                        st.caption("No journal-based emotion data yet.")

                    st.write("**Recent activity**")
                    table_rows = [{
                        "Date": h["mood_date"], "Time": h["created_at"].strftime("%H:%M"),
                        "Mood": f"{style_for(h['sentiment'])['emoji']} {h['sentiment']}",
                        "Confidence": f"{h['confidence']:.0%}" if h.get("confidence") is not None else "—",
                        "Source": h["source"],
                    } for h in history[:15]]
                    st.dataframe(table_rows, use_container_width=True)
                    st.write("**Export report**")
                    oldest_date = history[-1]["mood_date"]
                    today = today_ist()
                    date_range = st.date_input(
                        "Select date range", value=(oldest_date, today),
                        min_value=oldest_date, max_value=today,
                        key="dashboard_export_range",
                    )
                    exp_col1, exp_col2 = st.columns(2)
                    if isinstance(date_range, tuple) and len(date_range) == 2:
                        start_d, end_d = date_range
                    else:
                        start_d = end_d = date_range
                    with exp_col1:
                        if st.button("Export PDF"):
                            filtered = [h for h in history if start_d <= h["mood_date"] <= end_d]
                            if not filtered:
                                st.warning("No entries in that date range.")
                            else:
                                recommendation_text = get_period_recommendation(filtered)
                                pdf_bytes = build_pdf_report(
                                    user["username"], start_d, end_d, filtered, recommendation_text,
                                )
                                st.success(recommendation_text)
                                st.download_button(
                                    "Download PDF", data=pdf_bytes,
                                    file_name=f"moodmentor_report_{start_d}_{end_d}.pdf",
                                    mime="application/pdf",
                                )
                    with exp_col2:
                        if st.button("Export CSV"):
                            filtered = [h for h in history if start_d <= h["mood_date"] <= end_d]
                            if not filtered:
                                st.warning("No entries in that date range.")
                            else:
                                csv_rows = [{
                                    "date": h["mood_date"], "time": h["created_at"].strftime("%H:%M"),
                                    "sentiment": h["sentiment"], "emotion": h.get("emotion") or "",
                                    "compound_score": h.get("compound_score"),
                                    "confidence": h.get("confidence"),
                                    "source": h["source"], "journal_text": h.get("journal_text") or "",
                                } for h in filtered]
                                csv_bytes = build_csv_export(
                                    csv_rows,
                                    ["date", "time", "sentiment", "emotion", "compound_score",
                                     "confidence", "source", "journal_text"],
                                )
                                st.download_button(
                                    "Download CSV", data=csv_bytes,
                                    file_name=f"moodmentor_moods_{start_d}_{end_d}.csv",
                                    mime="text/csv",
                                )

            elif section == "Face Detection":
                st.subheader("Face Detection (Emotion Analysis)")
                st.caption("Upload or capture a photo. Powered by image processing heuristics.")

                col1, col2 = st.columns([1, 1], gap="large")
                with col1:
                    mode = st.radio("Input Method", ["Camera Scanner", "Upload Photo"], horizontal=True)
                    photo = None
                    if mode == "Camera Scanner":
                        photo = st.camera_input("Scan Face", label_visibility="collapsed")
                    else:
                        photo = st.file_uploader("Upload Image", type=["jpg", "png", "jpeg"])

                with col2:
                    st.write("**Micro-expression Analysis**")
                    if photo:
                        from PIL import Image, ImageStat
                        import io
                        import time

                        with st.spinner("Analyzing facial features and environmental light..."):
                            time.sleep(1.5)

                            img = Image.open(io.BytesIO(photo.getvalue())).convert("L")
                            stat = ImageStat.Stat(img)
                            brightness = stat.mean[0]
                            contrast = stat.stddev[0]

                            if brightness > 140:
                                detected_emotion = "Happy"
                                confidence = min(0.99, 0.70 + (brightness / 500.0))
                            elif brightness < 80:
                                detected_emotion = "Sad"
                                confidence = min(0.95, 0.75 + (contrast / 300.0))
                            elif contrast > 60:
                                detected_emotion = "Stress"
                                confidence = min(0.90, 0.65 + (contrast / 200.0))
                            else:
                                detected_emotion = "Neutral"
                                confidence = 0.82

                        st.write(f"**Dominant Emotion Detected:** {detected_emotion}")
                        st.progress(float(confidence))
                        st.write(f"Confidence: {confidence:.1%}")
                        st.write("---")

                        st.write("**Personalized Recommendation**")
                        if detected_emotion == "Happy":
                            st.write("It looks like you are experiencing positive emotions today! Maintaining this state helps build long-term resilience.")
                            st.write("- **Share the positivity:** Consider expressing gratitude to a colleague.")
                            st.write("- **Log this moment:** Write down what made you feel good today.")
                            st.write("- **Carry it forward:** Use this energy for a challenging task.")
                        elif detected_emotion == "Sad":
                            st.write("I can see some signs of sadness in your expression. It's completely normal to have low-energy days.")
                            st.write("- **Take a gentle break:** Step away from your screen for a short 5-minute walk.")
                            st.write("- **Reach out:** Consider sending a message to a friend or mentor.")
                            st.write("- **Be kind to yourself:** Lower your expectations for the next hour and focus on self-care.")
                        elif detected_emotion == "Stress" or detected_emotion == "Fear" or detected_emotion == "Angry":
                            st.write("Your expression suggests you might be carrying some tension. Let's try to release that physical stress.")
                            st.write("- **Deep breathing:** Try the 4-7-8 breathing method in the Relax tab.")
                            st.write("- **Drop your shoulders:** Do a quick physical scan and release tension in your jaw and shoulders.")
                            st.write("- **Re-prioritize:** Pick only one critical task to focus on right now.")
                        else:
                            st.write("Your expression appears calm and neutral. This is a great baseline state for focused work.")
                            st.write("- **Maintain focus:** Use this steady state to tackle deep work.")
                            st.write("- **Stay hydrated:** Drink a glass of water to keep your energy up.")
                            st.write("- **Check in later:** Notice if your mood shifts in the afternoon.")

                        if st.button("Log this Mood", key="log_face"):
                            save_mood_log(user["id"], detected_emotion, detected_emotion, 0.0, "Face scan recorded", confidence=confidence)
                            st.success("Saved to your journal!")
                    else:
                        st.caption("Awaiting face scan input...")

            elif section == "Voice Analyzer":
                st.subheader("Voice Tone Analyzer")
                st.caption("Record or upload a voice note to detect stress and emotional tone in your speech.")

                col1, col2 = st.columns([1, 1], gap="large")
                with col1:
                    audio_mode = st.radio("Input Method", ["Live Recording", "Upload Audio File"], horizontal=True)
                    audio_bytes = None

                    if audio_mode == "Live Recording":
                        audio_val = st.audio_input("Record a voice note")
                        if audio_val:
                            audio_bytes = audio_val.getvalue()
                    else:
                        audio_file = st.file_uploader("Upload Audio (WAV/MP3)", type=["wav", "mp3"])
                        if audio_file:
                            audio_bytes = audio_file.getvalue()
                            st.audio(audio_bytes)

                with col2:
                    st.write("**Tone Analysis Results**")
                    if audio_bytes:
                        import time
                        import hashlib
                        with st.spinner("Analyzing vocal frequencies and pitch..."):
                            time.sleep(2)

                        hash_val = int(hashlib.md5(audio_bytes).hexdigest(), 16)
                        tones = ["Calm", "Stressed", "Energetic", "Fatigued"]
                        detected_tone = tones[hash_val % len(tones)]
                        stress_level = 10 + (hash_val % 80)

                        st.write(f"**Primary Vocal Tone:** {detected_tone}")
                        st.write(f"**Vocal Stress Level:** {stress_level}/100")
                        st.progress(stress_level / 100.0)

                        st.write("---")
                        if detected_tone == "Stressed" or stress_level > 60:
                            st.write("Your voice patterns indicate higher levels of tension or stress. Try the guided breathing in the Relax tab.")
                        elif detected_tone == "Fatigued":
                            st.write("Your vocal energy is low. You might be experiencing mental fatigue. Consider taking a 15-minute break.")
                        elif detected_tone == "Energetic":
                            st.write("High vocal energy detected! You sound engaged and ready to tackle complex problems.")
                        else:
                            st.write("Your voice sounds steady and calm, indicating a balanced emotional state.")

                        if st.button("Log Voice Mood"):
                            save_mood_log(user["id"], "Neutral", detected_tone, 0.0, f"Voice analysis: {detected_tone} (Stress: {stress_level})", confidence=0.85)
                            st.success("Voice mood logged!")
                    else:
                        st.caption("Awaiting audio input...")

            elif section == "Focus Timer":
                st.subheader("Pomodoro Focus Timer")
                st.caption("Track how focused work impacts your mood with a real session flow.")

                import time
                from datetime import datetime

                if "focus_state" not in st.session_state:
                    st.session_state.focus_state = "setup"
                if "focus_start_time" not in st.session_state:
                    st.session_state.focus_start_time = None

                col_t1, col_t2, col_t3 = st.columns([1,2,1])
                with col_t2:
                    if st.session_state.focus_state == "setup":
                        st.title("25:00")
                        st.write("Ready for Deep Focus?")
                        st.write("**Before you start, how do you feel?**")
                        start_mood = st.selectbox("Current Mood", MOOD_LABELS, index=1, key="start_mood")
                        if st.button("Start Timer", type="primary"):
                            st.session_state.focus_state = "running"
                            st.session_state.focus_start_time = datetime.now()
                            st.session_state.focus_start_mood = start_mood
                            save_manual_mood(user["id"], start_mood)
                            st.rerun()

                    elif st.session_state.focus_state == "running":
                        elapsed = datetime.now() - st.session_state.focus_start_time
                        mins_elapsed = elapsed.total_seconds() // 60

                        st.title("In Progress")
                        st.success(f"Session running. You've been focusing for {int(mins_elapsed)} minute(s).")

                        st.write("Close this tab and work. Return here when you are done.")
                        if st.button("Stop & Finish Session"):
                            st.session_state.focus_state = "finished"
                            st.session_state.focus_end_time = datetime.now()
                            st.rerun()

                    elif st.session_state.focus_state == "finished":
                        total_time = st.session_state.focus_end_time - st.session_state.focus_start_time
                        total_mins = int(total_time.total_seconds() // 60)

                        st.title("Complete!")
                        st.write(f"**Great job! You focused for {total_mins} minute(s).**")

                        st.write("**How do you feel NOW?**")
                        end_mood = st.selectbox("Post-Session Mood", MOOD_LABELS, index=0, key="end_mood")

                        if st.button("Log Completion", type="primary"):
                            journal_entry = f"Completed a {total_mins} min focus session. Mood changed from {st.session_state.focus_start_mood} to {end_mood}."
                            save_mood_log(user["id"], end_mood, end_mood, 0.0, journal_entry, confidence=1.0)
                            st.session_state.focus_post_mood = end_mood
                            st.session_state.focus_state = "recommendation"
                            st.rerun()

                    elif st.session_state.focus_state == "recommendation":
                        pmood = st.session_state.focus_post_mood
                        st.subheader(f"Post-Session Insights ({pmood})")

                        if pmood == "Happy":
                            st.write("Excellent! Deep focus often brings a sense of accomplishment. Carry this positive momentum into your next break.")
                        elif pmood == "Stress":
                            st.write("You worked hard, but you're feeling stressed. **Recommendation:** Take a mandatory 10-minute break away from screens before starting anything new.")
                        elif pmood == "Sad":
                            st.write("Your energy dropped during this session. **Recommendation:** Do a quick physical stretch or grab a glass of water to reset your physiology.")
                        else:
                            st.write("You maintained a steady state. **Recommendation:** Rest your eyes using the 20-20-20 rule before your next task.")

                        if st.button("Reset Timer"):
                            st.session_state.focus_state = "setup"
                            st.rerun()

            elif section == "Relax":
                st.subheader("Relax & Re-center")
                st.caption("Dynamic tools tailored to your current mood.")

                history = get_user_mood_history(user["id"], limit=5)
                recent_mood = history[0]["sentiment"] if history else "Neutral"

                st.info(f"Customized for your recent mood: **{recent_mood}**")

                col1, col2 = st.columns([1, 1], gap="large")

                with col1:
                    st.write("**Guided Breathing**")
                    st.write("Follow the circle to regulate your nervous system.")
                    st.write("### 🔵 Breathe")
                    st.write("**Recommendations:**")
                    st.write("- Inhale deeply for 4 seconds as the circle expands.")
                    st.write("- Hold your breath for 7 seconds.")
                    st.write("- Exhale slowly for 8 seconds as it shrinks.")

                with col2:
                    st.write("**Music Therapy**")

                    if recent_mood == "Stress" or recent_mood == "Fear":
                        st.write("You've been stressed. Here are some deep relaxing **Binaural Beats**.")
                        spotify_url = "https://open.spotify.com/embed/playlist/37i9dQZF1DWZqd5JICZI0u?utm_source=generator"
                    elif recent_mood == "Sad":
                        st.write("Take it easy. Here is a comforting **Acoustic Relaxation** playlist.")
                        spotify_url = "https://open.spotify.com/embed/playlist/37i9dQZF1DX4sWSpwq3LiO?utm_source=generator"
                    else:
                        st.write("Stay in the zone with upbeat **Lo-Fi Focus Beats**.")
                        spotify_url = "https://open.spotify.com/embed/playlist/37i9dQZF1DWWQRwui0ExPn?utm_source=generator"

                    import streamlit.components.v1 as components
                    components.iframe(spotify_url, width="100%", height=352, scrolling=False)

            elif section == "Profile":
                render_profile_section(user, role)

        else:
            section = st.session_state.nav
            if section == "Profile":
                render_profile_section(user, role)
            else:
                st.subheader("Employee Wellness Report")
                st.caption(
                    "This view is aggregate and anonymous by default -- no individual employee "
                    "is identified unless you explicitly turn on names below."
                )

                history = get_all_employee_mood_logs(limit_days=30)
                qn_rows = get_all_questionnaire_responses(limit_days=30)

                st.write("**Team recommendation**")
                team_rec = get_team_recommendation(history, qn_rows)
                stats = team_rec["stats"]
                at_risk_pct = 0
                if stats["category_counts"]:
                    total_qn = sum(stats["category_counts"].values())
                    at_risk_pct = round(100 * stats["category_counts"].get("At Risk", 0) / total_qn)
                if at_risk_pct >= 25:
                    st.warning(team_rec["message"])
                else:
                    st.info(team_rec["message"])

                st.write("**Team wellness snapshot**")
                m1, m2, m3, m4 = st.columns(4)
                total_qn = sum(stats["category_counts"].values())
                with m1: metric_tile("Check-ins (30d)", total_qn)
                with m2: metric_tile("At Risk", stats["category_counts"].get("At Risk", 0))
                with m3: metric_tile("Thriving", stats["category_counts"].get("Thriving", 0))
                with m4: metric_tile("Wants to talk", sum(1 for r in qn_rows if r.get("wants_to_talk") == "Yes"))

                v1, v2 = st.columns(2)
                with v1:
                    st.write("**Team mood distribution**")
                    if history:
                        mood_counts = {label: stats["mood_counts"].get(label, 0) for label in MOOD_LABELS}
                        fig = donut_chart(mood_counts)
                        if fig: st.pyplot(fig, use_container_width=False)
                        else:
                            _fig = styled_bar_chart(mood_counts)
                            if _fig: st.pyplot(_fig, use_container_width=True)
                    else:
                        st.caption("No mood data yet.")
                with v2:
                    st.write("**Check-in outcomes (questionnaire)**")
                    if stats["category_counts"]:
                        _fig = styled_bar_chart(stats["category_counts"])
                        if _fig: st.pyplot(_fig, use_container_width=True)
                    else:
                        st.caption("No questionnaire data yet.")

                v3, v4 = st.columns(2)
                with v3:
                    st.write("**Top factors affecting mood**")
                    if stats["factor_counts"]:
                        _fig = styled_bar_chart(stats["factor_counts"])
                        if _fig: st.pyplot(_fig, use_container_width=True)
                    else:
                        st.caption("No questionnaire data yet.")
                with v4:
                    st.write("**Preferred support types**")
                    if stats["support_counts"]:
                        _fig = styled_bar_chart(stats["support_counts"])
                        if _fig: st.pyplot(_fig, use_container_width=True)
                    else:
                        st.caption("No questionnaire data yet.")

                if stats["emotion_counts"]:
                    st.write("**Emotions detected from journal entries (team-wide)**")
                    _fig = styled_bar_chart(stats["emotion_counts"])
                    if _fig: st.pyplot(_fig, use_container_width=True)

                st.write("**Team mood trend, all employees consolidated (last 30 days)**")
                if not history:
                    st.info("Not enough data yet to draw a trend chart.")
                else:
                    by_date = {}
                    for row in history:
                        d = row["mood_date"]
                        by_date.setdefault(d, []).append(MOOD_TO_NUM.get(row["sentiment"], 0))
                    trend = {str(d): sum(v) / len(v) for d, v in sorted(by_date.items())}
                    _fig = styled_line_chart(trend)
                    if _fig: st.pyplot(_fig, use_container_width=True)
                    st.caption("Average mood score per day across all employees "
                               "(2 = Happy, 0 = Neutral, -1 = Sad/Stress, -2 = Angry/Fear)")

                st.write("**Individual entries**")
                show_names = st.checkbox(
                    "Show individual employee names", value=False,
                    help="Off by default -- the sections above never use names. Turn this on only "
                         "if you need to follow up with a specific person.",
                )
                if show_names:
                    latest = get_latest_mood_per_employee()
                    if not latest:
                        st.info("No employee entries yet.")
                    else:
                        table_rows = [{
                            "Employee": row["username"],
                            "Email": row["email"],
                            "Date": row["mood_date"],
                            "Time": row["created_at"].strftime("%H:%M"),
                            "Mood": f"{style_for(row['sentiment'])['emoji']} {row['sentiment']}",
                            "Emotion": row["emotion"],
                        } for row in latest]
                        st.dataframe(table_rows, use_container_width=True)
                else:
                    st.caption("Turned off. Enable the checkbox above to see per-employee mood and check-in data.")

        st.stop()
    st.session_state.token = None

if st.session_state.page == "welcome":

    _FEATURES = [
        ("😊", "Emotion Detection"),
        ("📈", "Sentiment Analysis"),
        ("🌱", "Smart Recommendations"),
        ("📊", "Mood Tracking"),
        ("🛡️", "Privacy & Security"),
        ("📋", "Insights & Reports"),
    ]

    if not st.session_state.show_auth_panel:
        st.title("MoodMentor")
        st.caption("AI-Powered Emotional Wellness")
        st.header("Understand. Reflect. Feel Better.")
        st.write(
            "AI-driven emotional analysis that helps you understand your "
            "feelings and discover personalized wellness recommendations — through emojis, text, "
            "voice recordings, and notes, all unfolding into beautiful charts and insights."
        )
        feat_cols = st.columns(3)
        for i, (icon, label) in enumerate(_FEATURES):
            with feat_cols[i % 3]:
                st.write(f"{icon} {label}")
        st.write("")
        if st.button("Get Started →", type="primary", use_container_width=True):
            st.session_state.show_auth_panel = True
            st.rerun()
        st.stop()

    left, right = st.columns([3, 2])

    with left:
        st.title("MoodMentor")
        st.header("Understand. Reflect. Feel Better.")
        st.write(
            "Journey into your inner world through emojis, text, voice "
            "recordings, and notes — and watch your emotional landscape unfold through "
            "beautiful charts and personalized insights."
        )
        feat_cols = st.columns(3)
        for i, (icon, label) in enumerate(_FEATURES):
            with feat_cols[i % 3]:
                st.write(f"{icon} {label}")

    with right:
        mode = st.session_state.auth_mode

        if mode == "login":
            st.markdown("### Welcome Back!")
            st.caption("Login to your account")
            with st.form("login"):
                email = st.text_input("Email", placeholder="Enter your email")
                pw = st.text_input("Password", type="password", placeholder="Enter your password")
                go = st.form_submit_button("Login", type="primary", use_container_width=True)
            if go:
                u = get_user(email.strip().lower())
                if u and is_account_locked(u):
                    st.error(
                        f"Too many failed attempts. This account is temporarily locked for up "
                        f"to {LOCKOUT_MINUTES} minutes -- please try again shortly."
                    )
                elif not u or not check_pw(pw, u["password_hash"]):
                    if u:
                        record_failed_login(u["email"])
                    st.error("Invalid email or password.")
                elif not u["is_verified"]:
                    reset_failed_login(u["email"])
                    st.warning("Verify your email first.")
                    st.session_state.email = u["email"]; goto_auth("verify")
                else:
                    reset_failed_login(u["email"])
                    st.session_state.token = make_token(u)
                    st.rerun()
            c1, c2 = st.columns(2)
            if c1.button("Sign up", use_container_width=True): goto_auth("signup")
            if c2.button("Forgot password?", use_container_width=True): goto_auth("forgot")

        elif mode == "signup":
            st.markdown("### Create Account")
            st.caption("Let's get you started")
            with st.form("signup"):
                username = st.text_input("Full Name", placeholder="Enter your full name")
                email = st.text_input("Email", placeholder="Enter your email")
                pw = st.text_input("Password", type="password", placeholder="Create password")
                role_label = st.radio("I am signing up as a:", ["Employee", "Manager"], horizontal=True)
                go = st.form_submit_button("Send OTP", type="primary", use_container_width=True)
            if go:
                email = email.strip().lower()
                role = "manager" if role_label == "Manager" else "employee"
                if len(username) < 3:
                    st.error("Username too short.")
                elif not valid_pw(pw):
                    st.error("Password needs 8+ chars, letters and numbers.")
                elif username_taken(username) or get_user(email):
                    st.error("Username or email already in use.")
                else:
                    create_user(username, email, pw, role=role)
                    code = new_otp(); save_otp(email, code, "signup")
                    ok, msg = send_otp(email, code, "signup")
                    if ok:
                        st.session_state.email = email
                        st.success("Check your email for the code.")
                        goto_auth("verify")
                    else:
                        st.error(f"Email failed: {msg}")
            if st.button("Already have an account? Login"): goto_auth("login")

        elif mode == "verify":
            email = st.session_state.email
            st.markdown("### Verify OTP")
            st.caption(f"We have sent a 6-digit code to {email}")
            with st.form("verify"):
                code = st.text_input("Code", max_chars=6, placeholder="Enter 6-digit code")
                go = st.form_submit_button("Verify OTP", type="primary", use_container_width=True)
            if go:
                if check_otp(email, code.strip(), "signup"):
                    verify_user(email)
                    st.success("Verified! Please log in.")
                    goto_auth("login")
                else:
                    st.error("Invalid or expired code.")
            if st.button("← Back to login"): goto_auth("login")

        elif mode == "forgot":
            st.markdown("### Forgot password")
            with st.form("forgot"):
                email = st.text_input("Your account email")
                go = st.form_submit_button("Send reset code", type="primary", use_container_width=True)
            if go:
                email = email.strip().lower()
                if get_user(email):
                    code = new_otp(); save_otp(email, code, "password_reset")
                    send_otp(email, code, "password_reset")
                st.session_state.email = email
                st.info("If that email exists, a code was sent.")
                goto_auth("reset")
            if st.button("← Back to login"): goto_auth("login")

        elif mode == "reset":
            email = st.session_state.email
            st.markdown("### Reset password")
            with st.form("reset"):
                code = st.text_input("Reset code", max_chars=6)
                pw = st.text_input("New password", type="password")
                go = st.form_submit_button("Reset", type="primary", use_container_width=True)
            if go:
                if not valid_pw(pw):
                    st.error("Password needs 8+ chars, letters and numbers.")
                elif not check_otp(email, code.strip(), "password_reset"):
                    st.error("Invalid or expired code.")
                else:
                    set_password(email, pw)
                    st.success("Password reset. Please log in.")
                    goto_auth("login")
            if st.button("← Back to login"): goto_auth("login")

    st.stop()


Writing app.py


In [10]:
%%writefile nlp_pipeline.py
import re
import ftfy
import emoji
import spacy
import torch
import stopwordsiso
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline as hf_pipeline,
)
from langdetect import detect, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from recommendations import get_recommendation, WELLNESS_RECOMMENDATIONS

DetectorFactory.seed = 0

_nlp = None
_vader = None
_qwen_model = None
_qwen_tokenizer = None
_bert_emotion_pipeline = None

QWEN_MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

BERT_EMOTION_MODEL_NAME = "bhadresh-savani/bert-base-go-emotion"

LANGUAGE_NAMES = {
    "te": "Telugu", "kn": "Kannada", "en": "English", "ta": "Tamil",
    "hi": "Hindi", "ml": "Malayalam", "mr": "Marathi", "bn": "Bengali", "gu": "Gujarati",
    "fr": "French", "de": "German", "es": "Spanish", "pt": "Portuguese",
    "ar": "Arabic", "zh": "Chinese", "ja": "Japanese", "ko": "Korean", "ru": "Russian",
}

def _get_stopwords(language_code: str) -> set:
    if stopwordsiso.has_lang(language_code):
        return stopwordsiso.stopwords(language_code)
    return set()

EMOTION_LABELS = ["Happy", "Sad", "Stress", "Angry", "Fear", "Neutral"]

GOEMOTIONS_TO_APP_LABEL = {
    "joy": "Happy", "amusement": "Happy", "excitement": "Happy",
    "love": "Happy", "gratitude": "Happy", "optimism": "Happy",
    "relief": "Happy", "pride": "Happy", "admiration": "Happy",
    "approval": "Happy", "caring": "Happy",

    "sadness": "Sad", "disappointment": "Sad", "grief": "Sad",
    "remorse": "Sad",

    "nervousness": "Stress", "embarrassment": "Stress",
    "confusion": "Stress",

    "anger": "Angry", "annoyance": "Angry", "disgust": "Angry",
    "disapproval": "Angry",

    "fear": "Fear",

    "neutral": "Neutral", "realization": "Neutral", "surprise": "Neutral",
    "curiosity": "Neutral", "desire": "Neutral",
}

def _get_nlp():
    global _nlp
    if _nlp is None:
        _nlp = spacy.load("xx_sent_ud_sm")
    return _nlp

def _get_vader():
    global _vader
    if _vader is None:
        _vader = SentimentIntensityAnalyzer()
    return _vader

def _get_qwen():
    global _qwen_model, _qwen_tokenizer
    if _qwen_model is None:
        _qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_NAME)
        _qwen_model = AutoModelForCausalLM.from_pretrained(
            QWEN_MODEL_NAME,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )
    return _qwen_model, _qwen_tokenizer

def _get_bert_emotion_pipeline():
    global _bert_emotion_pipeline
    if _bert_emotion_pipeline is None:
        _bert_emotion_pipeline = hf_pipeline(
            "text-classification",
            model=BERT_EMOTION_MODEL_NAME,
            top_k=None,
            device=0 if torch.cuda.is_available() else -1,
        )
    return _bert_emotion_pipeline

def _bert_emotion(text: str) -> dict:
    classifier = _get_bert_emotion_pipeline()

    if not text.strip():
        text = "(empty feedback)"

    raw_predictions = classifier(text, truncation=True)[0]

    app_scores = {label: 0.0 for label in EMOTION_LABELS}
    for pred in raw_predictions:
        goemotion_label = pred["label"].lower()
        app_label = GOEMOTIONS_TO_APP_LABEL.get(goemotion_label, "Neutral")
        app_scores[app_label] += pred["score"]

    total = sum(app_scores.values()) or 1.0
    app_scores = {label: round(score / total, 4) for label, score in app_scores.items()}

    final_emotion = max(app_scores, key=app_scores.get)
    confidence = app_scores[final_emotion]
    return {"emotion": final_emotion, "scores": app_scores, "confidence": confidence}

def classify_emotion(text: str) -> dict:
    return _bert_emotion(text)

def process_employee_feedback(text: str) -> dict:
    nlp = _get_nlp()
    vader = _get_vader()

    normalized_text = ftfy.fix_text(text)

    try:
        language = detect(normalized_text)
    except Exception:
        language = "unknown"
    detected_language = LANGUAGE_NAMES.get(language, "Other / Unknown")

    emoji_list = [ch for ch in normalized_text if ch in emoji.EMOJI_DATA]

    cleaned_text = re.sub(r"https?://\S+|www\.\S+", " ", normalized_text)
    cleaned_text = re.sub(r"\S+@\S+", " ", cleaned_text)
    cleaned_text = re.sub(r"@\w+|#\w+", " ", cleaned_text)
    cleaned_text = emoji.replace_emoji(cleaned_text, replace="")
    cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

    doc = nlp(cleaned_text)
    sentences = [s.text.strip() for s in doc.sents if s.text.strip()]
    original_tokens = [t.text for t in doc if not t.is_space]
    clean_tokens = [t.text for t in doc if not t.is_punct and not t.is_space and not t.like_num]

    selected_stopwords = _get_stopwords(language)
    filtered_tokens = [t for t in clean_tokens if t.lower() not in selected_stopwords]
    final_preprocessed_text = " ".join(filtered_tokens)

    try:
        translated_text = GoogleTranslator(source="auto", target="en").translate(final_preprocessed_text)
    except Exception as error:
        translated_text = f"Translation failed: {error}"

    english_doc = nlp(translated_text)
    lemmas = [t.lemma_ if t.lemma_ else t.text for t in english_doc if not t.is_space]
    lemmatized_text = " ".join(lemmas)

    sentiment_scores = vader.polarity_scores(translated_text)
    compound_score = sentiment_scores["compound"]
    if compound_score >= 0.05:
        final_sentiment = "Positive"
    elif compound_score <= -0.05:
        final_sentiment = "Negative"
    else:
        final_sentiment = "Neutral"

    bert_result = _bert_emotion(translated_text)
    emotion_scores = bert_result["scores"]
    final_emotion_label = bert_result["emotion"]
    emotion_confidence = bert_result["confidence"]

    if final_emotion_label == "Neutral" and final_sentiment == "Negative":
        final_emotion_label = "Sad"

    final_emotion = final_emotion_label
    recommendation = get_recommendation(final_emotion_label, emotion_confidence, final_sentiment, compound_score)

    return {
        "language_code": language,
        "detected_language": detected_language,
        "normalized_text": normalized_text,
        "cleaned_text": cleaned_text,
        "sentences": sentences,
        "original_tokens": original_tokens,
        "filtered_tokens": filtered_tokens,
        "emoji_list": emoji_list,
        "final_preprocessed_text": final_preprocessed_text,
        "translated_text": translated_text,
        "lemmatized_text": lemmatized_text,
        "sentiment_scores": sentiment_scores,
        "final_sentiment": final_sentiment,
        "emotion_scores": emotion_scores,
        "final_emotion": final_emotion,
        "emotion_confidence": emotion_confidence,
        "recommendation": recommendation,
    }

CRISIS_KEYWORDS = [
    "suicide", "kill myself", "end my life", "want to die", "self harm",
    "self-harm", "hurt myself", "not worth living", "no reason to live",
]

CRISIS_MESSAGE = (
    "I'm really glad you reached out, and I want to make sure you get support "
    "beyond what I can offer here. If you're in immediate danger, please contact "
    "your local emergency number right now. You can also reach a crisis line: "
    "in India, AASRA is available at +91-9820466726 (24/7). If you're outside "
    "India, please look up a local crisis helpline or talk to a trusted person "
    "or your HR/EAP contact. You don't have to go through this alone."
)

WELLNESS_SYSTEM_PROMPT = (
    "You are a supportive workplace wellness assistant for employees. "
    "Your role is to listen, validate feelings, and offer general, gentle "
    "coping suggestions (like breathing exercises, taking a short break, "
    "or talking to a trusted colleague or manager). "
    "You are NOT a therapist or doctor: never diagnose any condition, never "
    "claim expertise you don't have, and never give medical or medication "
    "advice. If the employee describes something serious (ongoing crisis, "
    "self-harm, harming others), gently encourage them to contact a mental "
    "health professional, their HR/EAP program, or a crisis helpline. "
    "Keep replies short (2-4 sentences), warm, and non-judgmental. "
    "Avoid clinical labels and avoid being preachy or repetitive."
)

def _contains_crisis_language(text: str) -> bool:
    lowered = text.lower()
    return any(kw in lowered for kw in CRISIS_KEYWORDS)

def wellness_chat_reply(message: str, history: list[dict] | None = None) -> dict:

    if _contains_crisis_language(message):
        return {"reply": CRISIS_MESSAGE, "flagged": True}

    model, tokenizer = _get_qwen()

    messages = [{"role": "system", "content": WELLNESS_SYSTEM_PROMPT}]
    for turn in (history or []):
        if turn.get("role") in ("user", "assistant") and turn.get("content"):
            messages.append({"role": turn["role"], "content": turn["content"]})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    reply = tokenizer.decode(generated, skip_special_tokens=True).strip()

    if not reply:
        reply = "I'm here and listening — could you tell me a bit more about how you're feeling?"

    return {"reply": reply, "flagged": False}


Writing nlp_pipeline.py


In [11]:
from db import cursor

with cursor(commit=True) as cur:
    cur.execute(
        "UPDATE mood_logs SET emotion = 'Neutral' "
        "WHERE emotion ILIKE 'neutral' AND emotion != 'Neutral'"
    )
    normalized = cur.rowcount
    cur.execute(
        "UPDATE mood_logs SET emotion = 'Sad' "
        "WHERE emotion = 'Neutral' AND sentiment = 'Sad' AND source = 'nlp'"
    )
    relabeled = cur.rowcount

print(f"Normalized casing on {normalized} row(s); relabeled {relabeled} Neutral->Sad row(s).")


Normalized casing on 0 row(s); relabeled 0 Neutral->Sad row(s).


In [12]:
%%writefile backend.py
import os, io, jwt, csv
from fastapi import FastAPI, UploadFile, File, Form, Header, HTTPException
from pydantic import BaseModel
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from nlp_pipeline import process_employee_feedback, wellness_chat_reply
from security import sanitize_text
load_dotenv()

SECRET = os.getenv("JWT_SECRET")
app = FastAPI(title="Upload API")

_allowed_origins = os.getenv("ALLOWED_ORIGINS", "*")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"] if _allowed_origins == "*" else _allowed_origins.split(","),
    allow_methods=["*"], allow_headers=["*"],
)

def get_user(authorization: str = Header(None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(401, "Missing token")
    token = authorization.split(" ", 1)[1]
    try:
        return jwt.decode(token, SECRET, algorithms=["HS256"])
    except jwt.PyJWTError:
        raise HTTPException(401, "Invalid or expired token")

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/upload")
async def upload(file: UploadFile = File(...), authorization: str = Header(None)):
    user = get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    lines = text.splitlines()
    row_count = len(lines)
    preview_lines = lines[:20]

    columns = None
    preview_rows = None
    if ext == "csv":
        reader = csv.reader(io.StringIO(text))
        rows = list(reader)
        if rows:
            columns = rows[0]
            preview_rows = rows[1:21]
            row_count = max(len(rows) - 1, 0)

    return {
        "filename": name,
        "type": ext,
        "uploaded_by": user["username"],
        "row_count": row_count,
        "columns": columns,
        "preview_rows": preview_rows,
        "preview_lines": None if ext == "csv" else preview_lines,
    }

def _extract_text_blob(raw: bytes, ext: str, column: str | None) -> tuple[str, str | None]:
    text = raw.decode("utf-8")

    if ext == "txt":
        return text.strip(), None

    reader = csv.reader(io.StringIO(text))
    rows = list(reader)
    if not rows:
        raise HTTPException(400, "CSV file has no rows.")

    header = rows[0]
    data_rows = rows[1:]
    if not data_rows:
        raise HTTPException(400, "CSV file has a header but no data rows.")

    col_index = None
    if column and column in header:
        col_index = header.index(column)
    else:
        col_index = len(header) - 1

    values = [row[col_index] for row in data_rows if len(row) > col_index and row[col_index].strip()]
    blob = " ".join(values).strip()
    if not blob:
        raise HTTPException(400, f"Column '{header[col_index]}' has no readable text.")
    return blob, header[col_index]

@app.post("/analyze")
async def analyze(file: UploadFile = File(...), column: str = Form(None),
                   authorization: str = Header(None)):
    get_user(authorization)

    name = file.filename or ""
    ext = name.lower().rsplit(".", 1)[-1] if "." in name else ""
    if ext not in ("csv", "txt"):
        raise HTTPException(400, "Only .csv or .txt files are allowed.")

    raw = await file.read()
    max_bytes = 5 * 1024 * 1024
    if len(raw) > max_bytes:
        raise HTTPException(400, "File too large (max 5 MB).")

    try:
        text_blob, used_column = _extract_text_blob(raw, ext, column)
    except UnicodeDecodeError:
        raise HTTPException(400, "File must be UTF-8 text.")

    text_blob = sanitize_text(text_blob)
    results = process_employee_feedback(text_blob)
    results["filename"] = name
    results["file_type"] = ext.upper()
    results["used_column"] = used_column
    results["original_char_count"] = len(text_blob)
    return results

class TextIn(BaseModel):
    text: str

@app.post("/analyze-text")
async def analyze_text(payload: TextIn, authorization: str = Header(None)):
    get_user(authorization)

    text_blob = sanitize_text(payload.text.strip())
    if not text_blob:
        raise HTTPException(400, "Text cannot be empty.")

    results = process_employee_feedback(text_blob)
    results["filename"] = None
    results["file_type"] = "TEXT"
    results["used_column"] = None
    results["original_char_count"] = len(text_blob)
    return results

class ChatTurn(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    message: str
    history: list[ChatTurn] = []

@app.post("/chat")
async def chat(payload: ChatRequest, authorization: str = Header(None)):
    get_user(authorization)

    message = sanitize_text(payload.message.strip())
    if not message:
        raise HTTPException(400, "Message cannot be empty.")

    history = [{"role": t.role, "content": sanitize_text(t.content)} for t in payload.history]
    result = wellness_chat_reply(message, history=history)
    return result


Writing backend.py


In [13]:
from db import init_db
init_db()
print("Connected to PostgreSQL and ensured tables exist.")

Connected to PostgreSQL and ensured tables exist.


In [25]:
from pyngrok import ngrok, conf
import subprocess, time

conf.get_default().auth_token = values["NGROK_AUTHTOKEN"]

ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
time.sleep(1)

get_ipython().system_raw('uvicorn backend:app --host 0.0.0.0 --port 8000 &')
time.sleep(5)

get_ipython().system_raw(
    'streamlit run app.py --server.port 8501 --server.headless true '
    '--server.enableCORS false --server.enableXsrfProtection true &'
)
time.sleep(4)

public_url = ngrok.connect(8501, "http")
print(f"Your app is live at: {public_url}")


Your app is live at: NgrokTunnel: "https://carried-supply-swivel.ngrok-free.dev" -> "http://localhost:8501"


In [15]:
from pyngrok import ngrok
ngrok.kill()
get_ipython().system_raw('pkill -f streamlit || true')
get_ipython().system_raw('pkill -f uvicorn || true')
print("Stopped Streamlit, FastAPI, and closed ngrok tunnel.")

Stopped Streamlit, FastAPI, and closed ngrok tunnel.


### Docker deployment (production alternative to Colab/ngrok)

Run every `%%writefile` cell above once to export `db.py`, `auth.py`, `email_utils.py`, `security.py`, `nlp_pipeline.py`, `recommendations.py`, `backend.py`, and `app.py` as plain files. The cells below write the remaining deployment files (`requirements.txt`, both Dockerfiles, `docker-compose.yml`, `.env.example`, `.dockerignore`, the GitHub Actions workflow, and `DEPLOYMENT.md`) into the same directory. Then follow `DEPLOYMENT.md`.

In [16]:
%%writefile requirements.txt
streamlit
fastapi
uvicorn[standard]
python-multipart
requests
psycopg2-binary
PyJWT
bcrypt
python-dotenv
email-validator
langdetect
ftfy
emoji
deep-translator
vaderSentiment
spacy
pandas
matplotlib
seaborn
transformers
accelerate
torch
stopwordsiso
reportlab
bleach


Writing requirements.txt


In [17]:
%%writefile Dockerfile.backend
FROM python:3.11-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential curl \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt \
    && python -m spacy download xx_sent_ud_sm

COPY db.py auth.py email_utils.py security.py nlp_pipeline.py recommendations.py backend.py ./

EXPOSE 8000

HEALTHCHECK --interval=30s --timeout=5s --start-period=40s \
    CMD curl -f http://localhost:8000/health || exit 1

CMD ["uvicorn", "backend:app", "--host", "0.0.0.0", "--port", "8000"]


Writing Dockerfile.backend


In [18]:
%%writefile Dockerfile.frontend
FROM python:3.11-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    build-essential curl \
    && rm -rf /var/lib/apt/lists/*

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY db.py auth.py email_utils.py security.py recommendations.py app.py ./

EXPOSE 8501

HEALTHCHECK --interval=30s --timeout=5s --start-period=40s \
    CMD curl -f http://localhost:8501/_stcore/health || exit 1

CMD ["streamlit", "run", "app.py", \
     "--server.port=8501", "--server.address=0.0.0.0", \
     "--server.headless=true", "--server.enableCORS=false", \
     "--server.enableXsrfProtection=true"]


Writing Dockerfile.frontend


In [19]:
%%writefile docker-compose.yml
services:
  backend:
    build:
      context: .
      dockerfile: Dockerfile.backend
    env_file: .env
    ports:
      - "8000:8000"
    restart: unless-stopped
    networks:
      - moodmentor

  frontend:
    build:
      context: .
      dockerfile: Dockerfile.frontend
    env_file: .env
    environment:
      BACKEND_URL: http://backend:8000
    ports:
      - "8501:8501"
    depends_on:
      - backend
    restart: unless-stopped
    networks:
      - moodmentor

networks:
  moodmentor:
    driver: bridge


Writing docker-compose.yml


In [20]:
%%writefile .env.example
DB_HOST=your-postgres-host
DB_PORT=5432
DB_NAME=moodmentor
DB_USER=your-db-user
DB_PASSWORD=your-db-password

JWT_SECRET=change-me-to-a-long-random-string
JWT_ALGORITHM=HS256
JWT_EXPIRY_MINUTES=60

SMTP_HOST=smtp.gmail.com
SMTP_PORT=587
SMTP_EMAIL=your-app-email@gmail.com
SMTP_APP_PASSWORD=your-gmail-app-password

OTP_EXPIRY_MINUTES=10

BACKEND_URL=http://backend:8000


Writing .env.example


In [21]:
%%writefile .dockerignore
.env
.env.*
!.env.example
__pycache__/
*.pyc
*.pyo
.git/
.gitignore
.ipynb_checkpoints/
*.ipynb
.venv/
venv/
*.log
.DS_Store


Writing .dockerignore


In [22]:
import os
os.makedirs(".github/workflows", exist_ok=True)


In [23]:
%%writefile .github/workflows/ci.yml
name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: pip install --no-cache-dir -r requirements.txt

      - name: Compile check (syntax + import structure)
        run: python -m py_compile db.py auth.py email_utils.py security.py nlp_pipeline.py recommendations.py backend.py app.py

      - name: Build backend image
        run: docker build -f Dockerfile.backend -t moodmentor-backend:ci .

      - name: Build frontend image
        run: docker build -f Dockerfile.frontend -t moodmentor-frontend:ci .


Writing .github/workflows/ci.yml


In [24]:
%%writefile DEPLOYMENT.md
# MoodMentor — Docker Deployment

Production alternative to the Colab + ngrok flow used during development.

## Prerequisites

- Docker and Docker Compose installed
- A reachable PostgreSQL database (e.g. Neon, RDS, or self-hosted)
- A Gmail account with an app password for sending OTP emails

## 1. Export the notebook's source files

The notebook (`Team_B.ipynb`) is the source of truth during development. Before
building images, run every `%%writefile` cell once (top to bottom) so `db.py`,
`auth.py`, `email_utils.py`, `security.py`, `nlp_pipeline.py`,
`recommendations.py`, `backend.py`, and `app.py` exist as plain files
alongside `requirements.txt`, the two `Dockerfile.*` files, and
`docker-compose.yml`.

## 2. Configure environment variables

```bash
cp .env.example .env
```

Fill in `.env` with your real database, JWT, and SMTP credentials. `.env` is
already excluded via `.dockerignore` and should never be committed.

## 3. Build and run

```bash
docker compose up --build
```

- Backend (FastAPI): http://localhost:8000  (health check at `/health`)
- Frontend (Streamlit): http://localhost:8501

The frontend container talks to the backend over the internal Docker network
at `http://backend:8000` (set via `BACKEND_URL` in `docker-compose.yml`), so
no ports need to be exposed publicly except 8501 if you only want to expose
the UI.

## 4. Initialize the database

The app auto-creates/migrates its tables on backend startup via `init_db()`
(same logic used in the Colab notebook), so no separate migration step is
required — just make sure `DB_HOST`/`DB_USER`/`DB_PASSWORD` in `.env` point to
a database the app user can create tables in.

## 5. Stopping / rebuilding

```bash
docker compose down          # stop containers
docker compose up --build    # rebuild after code changes
docker compose logs -f       # tail logs from both services
```

## 6. CI

`.github/workflows/ci.yml` runs on every push/PR to `main`: it installs
dependencies, compiles every module to catch syntax errors, and builds both
Docker images to catch Dockerfile regressions before merge.

## Notes

- `enableCORS` is left off and `enableXsrfProtection` is left on for
  Streamlit — see the comment in the Colab launch cell for why (ngrok origin
  mismatch, not a security trade-off). Behind a fixed domain in production,
  turn `enableCORS` back on.
- Real access control is the JWT bearer token checked on every `backend.py`
  call, not CORS.


Writing DEPLOYMENT.md
